<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9_multiclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 9 (multiclass) — sample-only normalized hNPE–hNDE training

This is an **alternative** to `Exercise_9_Hybrid_NPE_NDE.ipynb`; the original exercise is not changed.  We keep exactly the same Gaussian simulator and defensive simulation design, but refactor the machine-learning experiment around one principle:

> Posterior, nuisance-posterior, and likelihood residuals are learned jointly from balanced simulator/proposal classes, with conditional mass normalization—but without an explicit bridge penalty and without giving the network any analytic density or simulator-derived residual coordinate.

The notebook contains two experiments.

1. **Nuisance marginalized implicitly.**  We compare the same structured three-class model trained with CE alone and with CE plus a cross-fitted conditional-normalization loss.
2. **Nuisance represented explicitly.**  A four-class structured ratio network learns the marginal-POI, conditional-nuisance, and likelihood residuals.  Its architecture enforces the known nuisance cancellation in $r_P=N/P$ and composes $r_{PN}=r_Pr_N$ exactly.

The likelihood proposal flows receive two learned, context-conditioned, full-support affine coupling layers before their spline stacks.  This is a generic flow architecture trained only by NLL on samples; no analytic Gaussian mean, likelihood, score, or handcrafted residual is used.  Wider spline support, more bins, tail-stable classifier inputs, zero-initialized ratio heads, rotating independent normalization banks, and a two-stage optimizer address the numerical issues seen in the first full run.

**Scope.**  The proposal flows are trained first and then frozen.  Joint training refers to the shared residual classifier; it is not end-to-end optimization through the proposal flows.  The analytic simulator likelihood and nuisance-marginal density appear only in validation.  Known design densities and learned proposal densities remain part of posterior reconstruction, but never enter classifier training.


In [ ]:
# ========================================================================
# Google Colab setup — safe to re-run; a no-op outside Colab.
# ========================================================================
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
USE_DRIVE = True

def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab")
    else:
        ROOT = Path("/content")
    ROOT.mkdir(parents=True, exist_ok=True)

    REPO_DIR = ROOT / "nsbi-lhc-toolkit"
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    WORK_DIR = REPO_DIR / "workshops" / "ml4hep_tifr"
    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)
    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )
    for import_dir in (REPO_DIR / "src", TUTORIAL_DIR):
        import_path = str(import_dir.resolve())
        if import_path not in sys.path:
            sys.path.insert(0, import_path)
    run(sys.executable, "-m", "pip", "install", "-q", "nflows==0.14")
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)
else:
    candidates = [Path.cwd(), Path.cwd() / "workshops" / "ml4hep_tifr_colab"]
    TUTORIAL_DIR = next(
        (candidate for candidate in candidates if (candidate / "utils_hnpe.py").exists()),
        None,
    )
    if TUTORIAL_DIR is None:
        raise FileNotFoundError(
            "Run from the repository root or workshops/ml4hep_tifr_colab."
        )
    if str(TUTORIAL_DIR.resolve()) not in sys.path:
        sys.path.insert(0, str(TUTORIAL_DIR.resolve()))


## What is actually being tested?

A balanced $K$-class classifier with logits $s_k(z)$ estimates

$$
  \log\frac{\pi_a(z)}{\pi_b(z)}=s_a(z)-s_b(z).
$$

Pairwise cycles vanish algebraically in any shared-logit model, even before training.  They are therefore never treated as a closure test.  This version contains no separate bridge-consistency penalty: nuisance-marginal and evidence consistency are evaluated only on held-out draws after training.

The non-tautological training question is whether every learned residual has the correct conditional mass.  The non-tautological validation questions are whether the posterior and likelihood routes agree with fresh simulator truth, whether their implied evidence is parameter independent on supported regions, and whether the amortized posterior is calibrated.

All classifier classes occur once per anchor group.  Complete groups—not individual class rows—are split into training and validation sets.  The network sees only standardized sample coordinates and class labels; proposal and analytic log densities are not inputs.


In [ ]:
import copy
import gc
import hashlib
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from scipy.special import logsumexp
from scipy.stats import multivariate_normal, norm, wasserstein_distance
from torch.utils.data import DataLoader, TensorDataset

from utils_dual_hnde import importance_tail_summary
from utils_hnpe import (
    sample_spline_flow,
    scalar_spline_flow_cdf,
    scalar_spline_flow_icdf,
    spline_flow_log_prob,
    train_spline_flow,
)
from utils_plotting import export_standalone_figure_script


SEED = 19092026
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


def set_torch_seed(seed):
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


# This committed configuration is the paper-quality full run.  FAST_MODE is
# a practical first Colab pass; SMOKE_MODE checks only the complete code path.
SMOKE_MODE = False
FAST_MODE = False
LOAD_IF_AVAILABLE = False

if SMOKE_MODE:
    RUN_TAG = "smoke"
    N_FLOW, N_CLASS = 2_000, 3_000
    SCALAR_FLOW_EPOCHS, VECTOR_FLOW_EPOCHS = 2, 2
    CLASS_EPOCHS, ENSEMBLE_SIZE = 4, 1
    N_NORM_TRAIN_GROUPS, N_NORM_TRAIN_INNER, N_NORM_TRAIN_BANKS = 32, 8, 1
    N_NORM_VALID_GROUPS, N_NORM_VALID_INNER, N_NORM_VALID_BANKS = 32, 16, 1
    N_DIAGNOSTIC_REFERENCE = 64
    N_FLOW_AUDIT_SAMPLES = 128
elif FAST_MODE:
    RUN_TAG = "fast"
    N_FLOW, N_CLASS = 35_000, 70_000
    SCALAR_FLOW_EPOCHS, VECTOR_FLOW_EPOCHS = 12, 18
    CLASS_EPOCHS, ENSEMBLE_SIZE = 28, 2
    N_NORM_TRAIN_GROUPS, N_NORM_TRAIN_INNER, N_NORM_TRAIN_BANKS = 512, 32, 4
    N_NORM_VALID_GROUPS, N_NORM_VALID_INNER, N_NORM_VALID_BANKS = 128, 64, 2
    N_DIAGNOSTIC_REFERENCE = 256
    N_FLOW_AUDIT_SAMPLES = 1_024
else:
    RUN_TAG = "full"
    N_FLOW, N_CLASS = 500_000, 750_000
    SCALAR_FLOW_EPOCHS, VECTOR_FLOW_EPOCHS = 50, 80
    CLASS_EPOCHS, ENSEMBLE_SIZE = 70, 4
    N_NORM_TRAIN_GROUPS, N_NORM_TRAIN_INNER, N_NORM_TRAIN_BANKS = 4_096, 64, 8
    N_NORM_VALID_GROUPS, N_NORM_VALID_INNER, N_NORM_VALID_BANKS = 512, 128, 4
    N_DIAGNOSTIC_REFERENCE = 768
    N_FLOW_AUDIT_SAMPLES = 8_192

N_GRID_REFERENCE = 48 if SMOKE_MODE else (128 if FAST_MODE else 256)
N_CALIBRATION_CONTEXTS = 20 if SMOKE_MODE else (200 if FAST_MODE else 2_000)
N_CALIBRATION_SAMPLES = 128 if SMOKE_MODE else (1_024 if FAST_MODE else 2_048)
N_FOUR_CALIBRATION_CONTEXTS = 8 if SMOKE_MODE else (100 if FAST_MODE else 500)
N_FOUR_CALIBRATION_MU = 16 if SMOKE_MODE else (128 if FAST_MODE else 256)
N_FOUR_CALIBRATION_ALPHA = 8 if SMOKE_MODE else (16 if FAST_MODE else 32)

MODEL_DIR = Path("models_exercise9_multiclass_v2_tail") / RUN_TAG
FIGURE_SCRIPT_DIR = Path("exercise9_multiclass_v2_figures_scripts") / RUN_TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_SCRIPT_DIR.mkdir(parents=True, exist_ok=True)


# These vector flows are completely sample-trained.  The first two coupling
# layers are learned full-support conditional affine maps; the following RQS
# stack has a much wider interval and enough bins to preserve central detail.
VECTOR_FLOW_MODEL_CONFIG = {
    "architecture_version": "full_support_affine_rqs_v2",
    "n_unbounded_affine_layers": 2,
    "affine_hidden_features": 256,
    "affine_hidden_layers": 2,
    "use_layer_permutations": False,
    "n_coupling_layers": 10,
    "hidden_features": 512,
    "hidden_layers": 4,
    "spline_num_bins": 80,
    "spline_tail_bound": 25.0,
    "activation": "silu",
    "dropout_probability": 0.0,
    "identity_initialization": True,
}
VECTOR_FLOW_TRAINING_CONFIG = {
    "batch_size": 1024,
    "n_epochs": VECTOR_FLOW_EPOCHS,
    "learning_rate": 3.0e-4,
    "min_learning_rate": 1.0e-6,
    "lr_scheduler_factor": 0.5,
    "lr_scheduler_patience": 3,
    "validation_fraction": 0.2,
    "patience": 15,
    "gradient_clip": 1.0,
    "weight_decay": 1.0e-6,
    "retain_initial_model": True,
}


# A single monotone scalar RQS remains the stable proposal architecture for
# q_P and q_N.  It is unchanged by the vector-tail refactor.
SCALAR_FLOW_MODEL_CONFIG = {
    "n_coupling_layers": 1,
    "hidden_features": 128,
    "hidden_layers": 3,
    "spline_num_bins": 32,
    "spline_tail_bound": 12.0,
    "dropout_probability": 0.0,
    "identity_initialization": True,
}
SCALAR_FLOW_TRAINING_CONFIG = {
    "batch_size": 1024,
    "n_epochs": SCALAR_FLOW_EPOCHS,
    "learning_rate": 3.0e-4,
    "min_learning_rate": 1.0e-7,
    "validation_fraction": 0.2,
    "patience": 10,
    "weight_decay": 1.0e-5,
    "gradient_clip": 1.0,
    "lr_scheduler_patience": 2,
    "retain_initial_model": True,
}


CLASS_MODEL_CONFIG = {
    "hidden_features": 256 if not SMOKE_MODE else 96,
    "residual_blocks": 4 if not SMOKE_MODE else 2,
    "dropout_probability": 0.0,
    "log_ratio_bound": 20.0,
    "architecture_version": "structured_ratio_heads_v2",
}
CLASS_TRAINING_CONFIG = {
    "batch_size_groups": 1024 if not SMOKE_MODE else 256,
    "constraint_batch_groups": 16 if not SMOKE_MODE else 8,
    "constraint_validation_chunk_groups": 16,
    "n_epochs": CLASS_EPOCHS,
    "ce_learning_rate": 3.0e-4,
    "normalized_learning_rate": 1.0e-4,
    "minimum_learning_rate": 1.0e-5,
    "weight_decay": 1.0e-5,
    "validation_fraction": 0.2,
    "patience": max(10, CLASS_EPOCHS // 4),
    "gradient_clip": 1.0,
    "warmup_epochs": min(2, max(1, CLASS_EPOCHS // 3)),
    "ramp_epochs": min(5, max(1, CLASS_EPOCHS // 2)),
}
LAMBDA_NORM_3 = 0.20
LAMBDA_NORM_4 = 0.15

SIMULATOR_SIGMA = np.array([0.95, 0.38, 0.30], dtype=float)
X_OBS = np.array([0.40, 1.35, 0.12], dtype=float)


def export_exercise9_multiclass_figure(fig, script_name):
    path = export_standalone_figure_script(
        fig, script_name=script_name, output_dir=FIGURE_SCRIPT_DIR
    )
    png_path = FIGURE_SCRIPT_DIR / f"{Path(script_name).stem}.png"
    pdf_path = FIGURE_SCRIPT_DIR / f"{Path(script_name).stem}.pdf"
    fig.savefig(png_path, dpi=220, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print("Exported:", path, png_path, pdf_path)
    return path


print(
    f"mode={RUN_TAG}, N_flow={N_FLOW:,}, N_class={N_CLASS:,}, "
    f"ensemble={ENSEMBLE_SIZE}"
)


## The unchanged Gaussian simulator and exact validation densities

We retain Exercise 9 exactly:

$$
\begin{aligned}
  x_1&\sim\mathcal N(\mu+0.8\alpha,0.95^2),\\
  x_2&\sim\mathcal N(0.72\mu^2-0.4\alpha,0.38^2),\\
  x_3&\sim\mathcal N(0.8\cos\mu+0.3\alpha,0.30^2),
\end{aligned}
$$

with $\rho_\mu=0.9\mathcal N(0,1.5^2)+0.1\mathcal N(0,4^2)$ and
$\rho_\alpha=0.9\mathcal N(0,1^2)+0.1\mathcal N(0,3^2)$.

When $\alpha$ is hidden, it can be integrated analytically for validation.  Writing
$b(\mu)=(\mu,0.72\mu^2,0.8\cos\mu)$ and $v=(0.8,-0.4,0.3)$,

$$
  p_m(x\mid\mu)=0.9\,\mathcal N(x;b,\Sigma_\epsilon+vv^T)
  +0.1\,\mathcal N(x;b,\Sigma_\epsilon+9vv^T).
$$

Neither this expression nor the explicit analytic likelihood is passed to a network.


In [ ]:
def _mixture_logpdf(values, core_sigma, broad_sigma):
    values = np.asarray(values, dtype=float)
    return logsumexp(
        np.stack([
            np.log(0.9) + norm.logpdf(values, 0.0, core_sigma),
            np.log(0.1) + norm.logpdf(values, 0.0, broad_sigma),
        ]),
        axis=0,
    )

def design_mu_logpdf(mu):
    return _mixture_logpdf(mu, 1.5, 4.0)

def design_alpha_logpdf(alpha):
    return _mixture_logpdf(alpha, 1.0, 3.0)

def design_logpdf(theta):
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    return design_mu_logpdf(theta[:, 0]) + design_alpha_logpdf(theta[:, 1])

def _sample_mixture(n, core_sigma, broad_sigma, rng):
    broad = rng.random(int(n)) < 0.1
    sigma = np.where(broad, broad_sigma, core_sigma)
    return rng.normal(0.0, sigma)

def sample_mu(n, rng):
    return _sample_mixture(n, 1.5, 4.0, rng).astype(np.float32)

def sample_alpha(n, rng):
    return _sample_mixture(n, 1.0, 3.0, rng).astype(np.float32)

def sample_design(n, rng):
    return np.column_stack([sample_mu(n, rng), sample_alpha(n, rng)]).astype(
        np.float32
    )

def simulator_base_mean(mu):
    mu = np.asarray(mu, dtype=float).ravel()
    return np.column_stack([mu, 0.72 * mu**2, 0.8 * np.cos(mu)])

def simulator_mean(theta):
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    mu, alpha = theta[:, 0], theta[:, 1]
    return simulator_base_mean(mu) + alpha[:, None] * np.array([0.8, -0.4, 0.3])

def simulate(theta, rng):
    mean = simulator_mean(theta)
    return (mean + rng.normal(size=mean.shape) * SIMULATOR_SIGMA).astype(
        np.float32
    )

def log_likelihood(x, theta):
    """Analytic truth used only in validation cells."""
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    x = np.atleast_2d(np.asarray(x, dtype=float))
    if len(x) == 1 and len(theta) > 1:
        x = np.repeat(x, len(theta), axis=0)
    if len(x) != len(theta):
        raise ValueError("x and theta must have matching rows or one x row.")
    return np.sum(norm.logpdf(x, simulator_mean(theta), SIMULATOR_SIGMA), axis=1)

_NOISE_COV = np.diag(SIMULATOR_SIGMA**2)
_ALPHA_LOADING = np.array([0.8, -0.4, 0.3])
_MARGINAL_COVS = [
    _NOISE_COV + sigma_alpha**2 * np.outer(_ALPHA_LOADING, _ALPHA_LOADING)
    for sigma_alpha in (1.0, 3.0)
]

def marginal_log_likelihood(x, mu):
    """Exact p_m(x|mu) after integrating the design nuisance mixture."""
    mu = np.asarray(mu, dtype=float).ravel()
    x = np.atleast_2d(np.asarray(x, dtype=float))
    if len(x) == 1 and len(mu) > 1:
        x = np.repeat(x, len(mu), axis=0)
    if len(x) != len(mu):
        raise ValueError("x and mu must have matching rows or one x row.")
    residual = x - simulator_base_mean(mu)
    components = [
        np.log(weight)
        + np.atleast_1d(
            multivariate_normal.logpdf(residual, mean=np.zeros(3), cov=cov)
        )
        for weight, cov in zip((0.9, 0.1), _MARGINAL_COVS)
    ]
    return np.atleast_1d(logsumexp(np.stack(components), axis=0))

def normalize_log_curve(log_density, grid):
    log_density = np.asarray(log_density, dtype=float)
    shift = float(np.max(log_density))
    density = np.exp(log_density - shift)
    integral = np.trapezoid(density, grid)
    return density / integral, shift + np.log(integral)

def normalize_log_surface(log_density, x_grid, y_grid):
    log_density = np.asarray(log_density, dtype=float)
    shift = float(np.max(log_density))
    density = np.exp(log_density - shift)
    integral = np.trapezoid(np.trapezoid(density, y_grid, axis=1), x_grid)
    return density / integral, shift + np.log(integral)

def integrated_absolute_error(reference, estimate, grid):
    return float(np.trapezoid(np.abs(reference - estimate), grid))

def js_distance_discrete(reference, estimate, floor=1.0e-15):
    reference = np.asarray(reference, dtype=float).ravel() + floor
    estimate = np.asarray(estimate, dtype=float).ravel() + floor
    reference /= reference.sum()
    estimate /= estimate.sum()
    middle = 0.5 * (reference + estimate)
    divergence = 0.5 * np.sum(reference * np.log(reference / middle))
    divergence += 0.5 * np.sum(estimate * np.log(estimate / middle))
    return float(np.sqrt(max(0.0, divergence)))

print("Observed x:", X_OBS)
print("Truth check, log p_m(x_obs|mu=1.2):", marginal_log_likelihood(X_OBS, [1.2])[0])



def sample_vector_flow_design(n_samples, rng, score_columns):
    """Empirically stratify vector-flow contexts without evaluating a density."""

    n_samples = int(n_samples)
    pool = sample_design(max(n_samples, 100_000), rng)
    selected_values = pool[:, list(score_columns)]
    center = np.median(selected_values, axis=0)
    mad = 1.4826 * np.median(np.abs(selected_values - center), axis=0)
    scale = np.where(mad > 1.0e-6, mad, selected_values.std(axis=0))
    scale = np.where(scale > 1.0e-6, scale, 1.0)
    score = np.max(np.abs((selected_values - center) / scale), axis=1)
    q90, q99 = np.quantile(score, [0.90, 0.99])
    strata = [score < q90, (score >= q90) & (score < q99), score >= q99]
    fractions = np.array([0.71, 0.21, 0.08])
    counts = np.floor(fractions * n_samples).astype(int)
    counts[0] += n_samples - counts.sum()
    pieces = []
    for mask, count in zip(strata, counts):
        candidates = np.flatnonzero(mask)
        pieces.append(
            pool[rng.choice(candidates, size=int(count), replace=count > len(candidates))]
        )
    result = np.concatenate(pieces, axis=0)
    return result[rng.permutation(len(result))].astype(np.float32)


def sample_tail_enriched_design(n_samples, rng, tail_fraction=0.25):
    """Sample constraint anchors with extra empirical tail coverage.

    The target condition is Z(c)=1 for every c, so changing only the anchor
    weighting does not change that population solution.  Selection uses
    sampled parameter ranks; no simulator density is evaluated.
    """

    n_samples = int(n_samples)
    n_tail = min(n_samples, max(1, int(round(tail_fraction * n_samples))))
    bulk = sample_design(n_samples - n_tail, rng)
    pool = sample_design(max(1_024, 12 * n_tail), rng)
    center = np.median(pool, axis=0)
    mad = 1.4826 * np.median(np.abs(pool - center), axis=0)
    scale = np.where(mad > 1.0e-6, mad, pool.std(axis=0))
    scale = np.where(scale > 1.0e-6, scale, 1.0)
    score = np.max(np.abs((pool - center) / scale), axis=1)
    tail_pool = np.flatnonzero(score >= np.quantile(score, 0.90))
    selected = rng.choice(tail_pool, size=n_tail, replace=len(tail_pool) < n_tail)
    combined = np.concatenate([bulk, pool[selected]], axis=0)
    return combined[rng.permutation(len(combined))].astype(np.float32)


## Structured sample-only multiclass trainer

The classifier uses no density values.  Its inputs are robustly standardized sampled coordinates followed by the invertible compression $t\mapsto\operatorname{asinh}(t)$; this is approximately linear in the core and grows only logarithmically in distant tails.

Each scalar ratio head is a pre-normalized residual MLP and is initialized to zero, so every residual begins at one.  A gentle $\tanh$ cap prevents unsupported inputs from producing arbitrarily large finite-precision weights; the notebook reports head saturation, which must remain negligible rather than being hidden.

For three classes ordered $(S,P,L)$, the network learns two heads $a(\mu,x)$ and $c(\mu,x)$ and returns

$$[s_S,s_P,s_L]=[a,0,a-c].$$

For four classes ordered $(S,N,P,L)$, it learns $a(\mu,x)$, $b(\mu,\alpha,x)$, and $c(\mu,\alpha,x)$ and returns

$$[s_S,s_N,s_P,s_L]=[a+b,a,0,a+b-c].$$

Thus $N/P=e^a$ is exactly independent of $\alpha$ and $S/P=e^{a+b}=(N/P)(S/N)$ at finite capacity.  This restriction follows from the class construction and uses no analytic simulator information.

Complete triplets/quartets remain grouped through the split and minibatches.  CE is proper and unmodified—there is no focal loss, label smoothing, or bridge term.  After two CE epochs, AdamW is reset at a lower learning rate while the normalization moment is ramped in.


In [ ]:
class PreNormResidualBlock(nn.Module):
    def __init__(self, width, dropout_probability=0.0):
        super().__init__()
        self.norm = nn.LayerNorm(int(width))
        self.linear_one = nn.Linear(int(width), int(width))
        self.linear_two = nn.Linear(int(width), int(width))
        self.dropout = nn.Dropout(float(dropout_probability))

    def forward(self, values):
        update = self.linear_one(self.norm(values))
        update = F.silu(update)
        update = self.dropout(update)
        update = self.linear_two(update)
        return values + update / math.sqrt(2.0)


class RatioHead(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_features,
        residual_blocks,
        dropout_probability,
    ):
        super().__init__()
        self.input_layer = nn.Linear(int(input_dim), int(hidden_features))
        self.blocks = nn.ModuleList([
            PreNormResidualBlock(hidden_features, dropout_probability)
            for _ in range(int(residual_blocks))
        ])
        self.final_norm = nn.LayerNorm(int(hidden_features))
        self.output_layer = nn.Linear(int(hidden_features), 1)
        nn.init.zeros_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)

    def forward(self, values):
        hidden = F.silu(self.input_layer(values))
        for block in self.blocks:
            hidden = block(hidden)
        return self.output_layer(F.silu(self.final_norm(hidden)))[:, 0]


class StructuredRatioNetwork(nn.Module):
    def __init__(
        self,
        input_dim,
        n_classes,
        hidden_features=256,
        residual_blocks=4,
        dropout_probability=0.0,
        log_ratio_bound=20.0,
        architecture_version=None,
    ):
        super().__init__()
        del architecture_version
        self.input_dim = int(input_dim)
        self.n_classes = int(n_classes)
        self.log_ratio_bound = float(log_ratio_bound)
        if (self.n_classes, self.input_dim) not in {(3, 4), (4, 5)}:
            raise ValueError("Expected (n_classes,input_dim)=(3,4) or (4,5).")
        common = dict(
            hidden_features=hidden_features,
            residual_blocks=residual_blocks,
            dropout_probability=dropout_probability,
        )
        a_input_dim = self.input_dim if self.n_classes == 3 else 4
        self.a_head = RatioHead(a_input_dim, **common)
        self.c_head = RatioHead(self.input_dim, **common)
        self.b_head = None
        if self.n_classes == 4:
            self.b_head = RatioHead(self.input_dim, **common)

    def raw_ratio_heads(self, values):
        if self.n_classes == 3:
            a_values = values
        else:
            # [mu, alpha, x1, x2, x3] -> [mu, x1, x2, x3].
            a_values = values[:, [0, 2, 3, 4]]
        heads = {"a": self.a_head(a_values), "c": self.c_head(values)}
        if self.b_head is not None:
            heads["b"] = self.b_head(values)
        return heads

    def bounded_ratio_heads(self, values):
        raw = self.raw_ratio_heads(values)
        bound = self.log_ratio_bound
        if bound <= 0.0:
            return raw
        return {name: bound * torch.tanh(value / bound) for name, value in raw.items()}

    def forward(self, values):
        heads = self.bounded_ratio_heads(values)
        a, c = heads["a"], heads["c"]
        zeros = torch.zeros_like(a)
        if self.n_classes == 3:
            return torch.stack([a, zeros, a - c], dim=1)
        b = heads["b"]
        return torch.stack([a + b, a, zeros, a + b - c], dim=1)


def _assert_finite(name, values, ndim=None):
    values = np.asarray(values)
    if ndim is not None and values.ndim != ndim:
        raise ValueError(f"{name} must have ndim={ndim}; got {values.shape}.")
    if not np.isfinite(values).all():
        raise ValueError(f"{name} contains non-finite values.")
    return values

def _install_nflows_rqs_float64_retry():
    """Retry only a failed float32 inverse-RQS kernel in float64.

    For a monotone rational-quadratic spline the inverse discriminant is
    non-negative analytically.  nflows 0.14 evaluates it in float32, where a
    nearly double root can acquire a tiny negative value by cancellation.  We
    retry the *same* spline tensors in float64: no row is dropped, clipped, or
    resampled.  The original float64 assertion remains the hard guard.
    """
    import functools
    import importlib
    import inspect
    from importlib.metadata import version
    import warnings

    nflows_version = version("nflows")
    if nflows_version != "0.14":
        raise RuntimeError(
            "This audited numerical guard requires nflows==0.14; "
            f"found {nflows_version}."
        )
    module = importlib.import_module(
        "nflows.transforms.splines.rational_quadratic"
    )
    original = module.rational_quadratic_spline
    if getattr(original, "_exercise9_float64_retry", False):
        return original
    signature = inspect.signature(original)

    @functools.wraps(original)
    def guarded(*args, **kwargs):
        inputs_fast = kwargs.get("inputs", args[0] if args else None)
        inverse_fast = kwargs.get(
            "inverse", args[4] if len(args) > 4 else False
        )
        if (
            inverse_fast
            and torch.is_tensor(inputs_fast)
            and inputs_fast.dtype == torch.float32
        ):
            guarded._float32_inverse_call_count += 1
            guarded._float32_inverse_values += int(inputs_fast.numel())
        try:
            return original(*args, **kwargs)
        except AssertionError as error32:
            bound = signature.bind(*args, **kwargs)
            inputs = bound.arguments["inputs"]
            inverse = bound.arguments.get(
                "inverse", signature.parameters["inverse"].default
            )
            if not inverse or inputs.dtype != torch.float32:
                raise

            floating = [
                value for value in (*args, *kwargs.values())
                if torch.is_tensor(value) and value.is_floating_point()
            ]
            if any(not bool(torch.isfinite(value).all()) for value in floating):
                raise FloatingPointError(
                    "Non-finite tensor reached the inverse RQS; refusing the "
                    "precision retry."
                ) from error32

            def to_float64(value):
                if torch.is_tensor(value) and value.is_floating_point():
                    return value.to(dtype=torch.float64)
                return value

            try:
                outputs64, logdet64 = original(
                    *(to_float64(value) for value in args),
                    **{
                        name: to_float64(value)
                        for name, value in kwargs.items()
                    },
                )
            except AssertionError as error64:
                raise RuntimeError(
                    "The inverse-RQS discriminant also failed in float64. "
                    "Refusing to clip or resample; retrain this flow."
                ) from error64
            if not (
                bool(torch.isfinite(outputs64).all())
                and bool(torch.isfinite(logdet64).all())
            ):
                raise FloatingPointError(
                    "The float64 inverse-RQS retry returned non-finite values."
                ) from error32

            guarded._float64_retry_count += 1
            guarded._float64_retry_values += int(inputs.numel())
            if guarded._float64_retry_count == 1:
                warnings.warn(
                    "nflows float32 inverse-RQS cancellation: retrying the "
                    "same spline call in float64.",
                    RuntimeWarning,
                    stacklevel=2,
                )
            outputs = outputs64.to(dtype=inputs.dtype)
            logdet = logdet64.to(dtype=inputs.dtype)
            if not (
                bool(torch.isfinite(outputs).all())
                and bool(torch.isfinite(logdet).all())
            ):
                raise FloatingPointError(
                    "Casting the inverse-RQS retry back to float32 "
                    "produced non-finite values."
                ) from error32
            return outputs, logdet

    guarded._exercise9_float64_retry = True
    guarded._float64_retry_count = 0
    guarded._float64_retry_values = 0
    guarded._float32_inverse_call_count = 0
    guarded._float32_inverse_values = 0
    guarded._float32_original = original
    module.rational_quadratic_spline = guarded

    # nflows 0.14's linear-tail helper resolves the module global above.  The
    # aliases cover any direct bounded-spline call without touching package
    # source or a checkpoint.
    importlib.import_module(
        "nflows.transforms.splines"
    ).rational_quadratic_spline = guarded
    importlib.import_module(
        "nflows.transforms.autoregressive"
    ).rational_quadratic_spline = guarded
    linear_tail = importlib.import_module(
        "nflows.transforms.splines"
    ).unconstrained_rational_quadratic_spline
    if linear_tail.__globals__.get("rational_quadratic_spline") is not guarded:
        raise RuntimeError(
            "The nflows 0.14 linear-tail inverse did not bind to the "
            "audited RQS guard."
        )
    return guarded


RQS_NUMERIC_GUARD = _install_nflows_rqs_float64_retry()


def _rqs_retry_count():
    return int(getattr(RQS_NUMERIC_GUARD, "_float64_retry_count", 0))


def _rqs_inverse_call_count():
    return int(
        getattr(RQS_NUMERIC_GUARD, "_float32_inverse_call_count", 0)
    )


def _draw_conditional(flow_pack, contexts, n_samples, seed):
    contexts = np.atleast_2d(np.asarray(contexts, dtype=np.float32))
    n_samples = int(n_samples)
    n_features = int(flow_pack["config"]["n_features"])
    set_torch_seed(seed)
    draws = sample_spline_flow(
        flow_pack, n_samples, context=contexts, batch_size=16_384
    )
    draws = np.asarray(draws, dtype=np.float32)
    if len(contexts) == 1:
        draws = draws[None, :, :]
    expected = (len(contexts), n_samples, n_features)
    if draws.shape != expected:
        raise RuntimeError(
            f"Unexpected conditional sample shape {draws.shape}; "
            f"expected {expected}."
        )
    return _assert_finite("conditional flow draws", draws, ndim=3)


def _audit_context_subset(flow_pack, contexts, seed, n_contexts=2_048):
    contexts = _assert_finite("flow audit contexts", contexts, ndim=2).astype(
        np.float32
    )
    if len(contexts) < n_contexts:
        raise ValueError("The flow audit needs at least n_contexts rows.")
    scaler = flow_pack["context_scaler"]
    standardized = (contexts - scaler.mean) / scaler.std
    extremeness = np.max(np.abs(standardized), axis=1)
    n_tail = min(512, n_contexts // 4)
    tail_index = np.argpartition(extremeness, -n_tail)[-n_tail:]
    available = np.setdiff1d(
        np.arange(len(contexts)), tail_index, assume_unique=False
    )
    rng = np.random.default_rng(int(seed))
    typical_index = rng.choice(
        available, size=n_contexts - n_tail, replace=False
    )
    return contexts[np.concatenate([typical_index, tail_index])]


def audit_scalar_flow(flow_pack, contexts, name, seed):
    """Active inverse/forward, density, finiteness, and RNG checks."""
    if int(flow_pack["config"]["n_features"]) != 1:
        raise ValueError("audit_scalar_flow requires a scalar target flow.")
    history = flow_pack.get("history", {})
    initial_values = history.get("initial_validation", [])
    selected_values = history.get("selected_validation", [])
    if initial_values and selected_values:
        initial_validation = float(initial_values[-1])
        selected_validation = float(selected_values[-1])
        print(
            f"{name} validation NLL: identity={initial_validation:.4f}, "
            f"selected={selected_validation:.4f}."
        )
        if selected_validation >= initial_validation - 1.0e-4:
            import warnings
            warnings.warn(
                f"{name} retained its context-independent identity "
                "baseline; numerical closure can still pass, but proposal "
                "fidelity must be treated as failed until PIT/ESS checks.",
                RuntimeWarning,
                stacklevel=2,
            )
    contexts = _audit_context_subset(flow_pack, contexts, seed)
    z_grid = np.arange(-6.0, 7.0, dtype=np.float64)
    probabilities = np.tile(norm.cdf(z_grid), len(contexts))
    repeated_contexts = np.repeat(contexts, len(z_grid), axis=0)
    retry_before = _rqs_retry_count()
    inverse_calls_before = _rqs_inverse_call_count()

    quantiles = scalar_spline_flow_icdf(
        flow_pack,
        probabilities,
        context=repeated_contexts,
        batch_size=8_192,
    ).reshape(len(contexts), len(z_grid))
    if not np.isfinite(quantiles).all():
        raise FloatingPointError(f"{name}: non-finite inverse-CDF values.")
    if not np.all(np.diff(quantiles, axis=1) > 0.0):
        raise RuntimeError(f"{name}: inverse CDF is not strictly monotone.")

    recovered_probability = scalar_spline_flow_cdf(
        flow_pack,
        quantiles.reshape(-1, 1),
        context=repeated_contexts,
        batch_size=8_192,
    )
    recovered_z = norm.ppf(
        np.clip(
            recovered_probability,
            np.nextafter(0.0, 1.0),
            np.nextafter(1.0, 0.0),
        )
    ).reshape(quantiles.shape)
    z_error = np.abs(recovered_z - z_grid[None, :])
    q99_error = float(np.quantile(z_error, 0.99))
    max_error = float(np.max(z_error))
    if q99_error > 1.0e-4 or max_error > 2.0e-3:
        raise RuntimeError(
            f"{name}: inverse/forward closure failed: "
            f"q99={q99_error:.3g}, max={max_error:.3g}."
        )

    log_density = spline_flow_log_prob(
        flow_pack,
        quantiles.reshape(-1, 1),
        context=repeated_contexts,
        batch_size=8_192,
    )
    if not np.isfinite(log_density).all():
        raise FloatingPointError(f"{name}: non-finite density at its quantiles.")

    first = _draw_conditional(flow_pack, contexts, 2, seed + 1)
    second = _draw_conditional(flow_pack, contexts, 2, seed + 1)
    if not np.array_equal(first, second):
        raise RuntimeError(f"{name}: repeated seeded draws are not identical.")
    retry_delta = _rqs_retry_count() - retry_before
    inverse_call_delta = _rqs_inverse_call_count() - inverse_calls_before
    if retry_delta:
        raise RuntimeError(
            f"{name}: {retry_delta}/{inverse_call_delta} inverse-RQS "
            "kernel calls needed float64 in the numerical preflight. "
            "Retrain this scalar flow; reserve the fallback for a rare "
            "event in the much larger production draw."
        )
    print(
        f"{name} scalar-flow audit passed: q99 |z_back-z|={q99_error:.2e}, "
        f"max={max_error:.2e}, retried RQS kernels="
        f"{retry_delta}/{inverse_call_delta}."
    )


def _logmeanexp_torch(values, dim=-1):
    return torch.logsumexp(values, dim=dim) - math.log(values.shape[dim])


def _fit_classifier_transform(points):
    points = np.asarray(points, dtype=np.float32)
    center = np.median(points, axis=0).astype(np.float32)
    mad = np.median(np.abs(points - center), axis=0).astype(np.float32)
    robust_scale = (1.4826 * mad).astype(np.float32)
    ordinary_scale = points.std(axis=0, dtype=np.float64).astype(np.float32)
    scale = np.where(robust_scale > 1.0e-6, robust_scale, ordinary_scale)
    scale = np.where(scale > 1.0e-6, scale, 1.0).astype(np.float32)
    return center, scale


def _transform_classifier_points(points, center, scale):
    points = np.asarray(points, dtype=np.float32)
    return np.arcsinh((points - center) / scale).astype(np.float32)


def _as_constraint_banks(banks):
    if banks is None:
        return []
    return list(banks) if isinstance(banks, (list, tuple)) else [banks]


def _prepare_constraint_banks(banks, center, scale):
    prepared_banks = []
    for bundle in _as_constraint_banks(banks):
        prepared = {}
        for key, value in bundle.items():
            array = np.asarray(value, dtype=np.float32)
            if key.endswith("_points"):
                array = _transform_classifier_points(array, center, scale)
            prepared[key] = torch.as_tensor(array, dtype=torch.float32)
        prepared_banks.append(prepared)
    return prepared_banks


def _constraint_scale(epoch, config):
    warmup = int(config["warmup_epochs"])
    ramp = int(config["ramp_epochs"])
    if epoch < warmup:
        return 0.0
    return min(1.0, (epoch - warmup + 1) / max(1, ramp))


def _make_model(input_dim, n_classes):
    return StructuredRatioNetwork(
        input_dim=input_dim,
        n_classes=n_classes,
        **CLASS_MODEL_CONFIG,
    ).to(device)


def _multiclass_fingerprint(
    class_points,
    *,
    n_classes,
    seed_base,
    lambda_norm,
    constraint_train,
    constraint_validation,
):
    digest = hashlib.sha256()
    configuration = json.dumps(
        {
            "n_classes": int(n_classes),
            "seed_base": int(seed_base),
            "ensemble_size": int(ENSEMBLE_SIZE),
            "model": CLASS_MODEL_CONFIG,
            "training": CLASS_TRAINING_CONFIG,
            "lambda_norm": float(lambda_norm),
            "input_transform": "robust_asinh_v1",
            "normalization_estimator": "independent_cross_mass_selection_v2",
        },
        sort_keys=True,
        separators=(",", ":"),
    )
    digest.update(configuration.encode("utf-8"))

    def update_array(name, value):
        array = np.ascontiguousarray(value)
        digest.update(name.encode("utf-8"))
        digest.update(str(array.shape).encode("ascii"))
        digest.update(str(array.dtype).encode("ascii"))
        digest.update(array.view(np.uint8))

    update_array("class_points", class_points)
    for collection_name, collection in [
        ("constraint_train", constraint_train),
        ("constraint_validation", constraint_validation),
    ]:
        banks = _as_constraint_banks(collection)
        if not banks:
            digest.update(f"{collection_name}:none".encode("utf-8"))
        for bank_index, bundle in enumerate(banks):
            for key in sorted(bundle):
                update_array(f"{collection_name}:{bank_index}:{key}", bundle[key])
    return digest.hexdigest()


def _validation_ce(model, points_tensor, n_classes, batch_groups=2048):
    model.eval()
    total, count = 0.0, 0
    labels_one = torch.arange(n_classes, device=device)
    with torch.no_grad():
        for start in range(0, len(points_tensor), batch_groups):
            batch = points_tensor[start : start + batch_groups].to(device)
            logits = model(batch.reshape(-1, batch.shape[-1]))
            labels = labels_one.repeat(len(batch))
            total += float(F.cross_entropy(logits, labels, reduction="sum").cpu())
            count += len(labels)
    return total / max(1, count)


def _constraint_logits(model, bundle, key, index, n_classes):
    points = bundle[key].index_select(0, index).to(device)
    logits = model(points.reshape(-1, points.shape[-1]))
    return logits.reshape(*points.shape[:-1], int(n_classes))


def _partition_cross_mass(logits_a, logits_b, numerator, denominator):
    log_w_a = logits_a[..., numerator] - logits_a[..., denominator]
    log_w_b = logits_b[..., numerator] - logits_b[..., denominator]
    log_z_a = _logmeanexp_torch(log_w_a, dim=1)
    log_z_b = _logmeanexp_torch(log_w_b, dim=1)

    # The reduction happens in float32; only the cheap exponentiation and
    # cross product are promoted.  Independent A/B samples make the cross
    # product unbiased for (Z-1)^2.  A single minibatch may be negative.
    delta_a = torch.expm1(log_z_a.double())
    delta_b = torch.expm1(log_z_b.double())
    cross = (delta_a * delta_b).mean()
    pooled_monitor = (0.5 * (delta_a + delta_b)).square().mean()
    if not torch.isfinite(cross) or not torch.isfinite(pooled_monitor):
        raise FloatingPointError("Non-finite conditional mass estimate.")
    return cross.float(), pooled_monitor.float()


def _validation_mass_metrics(model, banks, loss_fn):
    if not banks:
        return 0.0, 0.0
    chunk_size = int(CLASS_TRAINING_CONFIG["constraint_validation_chunk_groups"])
    cross_sum, pooled_sum, count = 0.0, 0.0, 0
    model.eval()
    with torch.no_grad():
        for bundle in banks:
            n_groups = next(iter(bundle.values())).shape[0]
            for start in range(0, n_groups, chunk_size):
                stop = min(n_groups, start + chunk_size)
                index = torch.arange(start, stop, dtype=torch.long)
                cross, pooled = loss_fn(model, bundle, index)
                weight = stop - start
                cross_sum += float(cross.cpu()) * weight
                pooled_sum += float(pooled.cpu()) * weight
                count += weight
    return cross_sum / max(1, count), pooled_sum / max(1, count)


def _new_classifier_optimizer(model, learning_rate):
    return torch.optim.AdamW(
        model.parameters(),
        lr=float(learning_rate),
        betas=(0.9, 0.99),
        weight_decay=float(CLASS_TRAINING_CONFIG["weight_decay"]),
    )


def train_multiclass_ensemble(
    class_points,
    *,
    n_classes,
    checkpoint_dir,
    seed_base,
    constraint_train=None,
    constraint_validation=None,
    constraint_loss_fn=None,
    lambda_norm=0.0,
):
    class_points = _assert_finite("class_points", class_points, ndim=3).astype(np.float32)
    if class_points.shape[1] != n_classes:
        raise ValueError("class_points must contain one row per class per group.")
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    fingerprint = _multiclass_fingerprint(
        class_points,
        n_classes=n_classes,
        seed_base=seed_base,
        lambda_norm=lambda_norm,
        constraint_train=constraint_train,
        constraint_validation=constraint_validation,
    )

    split_rng = np.random.default_rng(SEED + 700 + n_classes)
    order = split_rng.permutation(len(class_points))
    n_validation = max(1, int(CLASS_TRAINING_CONFIG["validation_fraction"] * len(order)))
    validation_index, training_index = order[:n_validation], order[n_validation:]
    if set(validation_index).intersection(set(training_index)):
        raise RuntimeError("Group split leakage detected.")

    training_flat = class_points[training_index].reshape(-1, class_points.shape[-1])
    center, robust_scale = _fit_classifier_transform(training_flat)
    train_tensor = torch.as_tensor(
        _transform_classifier_points(class_points[training_index], center, robust_scale),
        dtype=torch.float32,
    )
    validation_tensor = torch.as_tensor(
        _transform_classifier_points(class_points[validation_index], center, robust_scale),
        dtype=torch.float32,
    )
    train_dataset = TensorDataset(train_tensor)
    prepared_train = _prepare_constraint_banks(constraint_train, center, robust_scale)
    prepared_validation = _prepare_constraint_banks(constraint_validation, center, robust_scale)

    ensemble = []
    labels_one = torch.arange(n_classes, device=device)
    for member in range(ENSEMBLE_SIZE):
        checkpoint = checkpoint_dir / f"member_{member}.pt"
        if LOAD_IF_AVAILABLE and checkpoint.exists():
            try:
                saved = torch.load(checkpoint, map_location=device, weights_only=False)
            except TypeError:
                saved = torch.load(checkpoint, map_location=device)
            if saved.get("fingerprint") != fingerprint:
                raise RuntimeError(
                    f"Checkpoint {checkpoint} belongs to different data, "
                    "architecture, loss, or normalization draws."
                )
            model = _make_model(class_points.shape[-1], n_classes)
            model.load_state_dict(saved["state_dict"])
            model.eval()
            ensemble.append({
                "model": model,
                "center": np.asarray(saved["center"], dtype=np.float32),
                "scale": np.asarray(saved["scale"], dtype=np.float32),
                "history": saved.get("history", {}),
                "checkpoint": checkpoint,
            })
            print("Loaded", checkpoint)
            continue

        member_seed = int(seed_base + member)
        set_torch_seed(member_seed)
        data_generator = torch.Generator().manual_seed(member_seed + 10_000)
        constraint_generator = torch.Generator().manual_seed(member_seed + 20_000)
        train_loader = DataLoader(
            train_dataset,
            batch_size=int(CLASS_TRAINING_CONFIG["batch_size_groups"]),
            shuffle=True,
            drop_last=False,
            generator=data_generator,
        )
        model = _make_model(class_points.shape[-1], n_classes)
        optimizer = _new_classifier_optimizer(
            model, CLASS_TRAINING_CONFIG["ce_learning_rate"]
        )
        scheduler = None
        history = {key: [] for key in [
            "train_ce", "validation_ce", "validation_mass_cross",
            "validation_mass_pooled", "validation_total",
            "constraint_scale", "learning_rate",
        ]}
        best_state, best_value, best_record, stale = None, math.inf, None, 0

        for epoch in range(CLASS_EPOCHS):
            if epoch == int(CLASS_TRAINING_CONFIG["warmup_epochs"]):
                optimizer = _new_classifier_optimizer(
                    model, CLASS_TRAINING_CONFIG["normalized_learning_rate"]
                )
                scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer,
                    mode="min",
                    factor=0.3,
                    patience=max(2, CLASS_EPOCHS // 10),
                    min_lr=float(CLASS_TRAINING_CONFIG["minimum_learning_rate"]),
                )

            model.train()
            scale_now = _constraint_scale(epoch, CLASS_TRAINING_CONFIG)
            bank = prepared_train[epoch % len(prepared_train)] if prepared_train else None
            ce_sum, n_rows = 0.0, 0
            for (group_batch,) in train_loader:
                group_batch = group_batch.to(device)
                logits = model(group_batch.reshape(-1, group_batch.shape[-1]))
                labels = labels_one.repeat(len(group_batch))
                ce = F.cross_entropy(logits, labels)
                mass_loss = torch.zeros((), device=device)
                if bank is not None and scale_now > 0.0:
                    n_groups = next(iter(bank.values())).shape[0]
                    index = torch.randint(
                        n_groups,
                        (min(int(CLASS_TRAINING_CONFIG["constraint_batch_groups"]), n_groups),),
                        generator=constraint_generator,
                    )
                    mass_loss, _ = constraint_loss_fn(model, bank, index)
                objective = ce + scale_now * float(lambda_norm) * mass_loss
                if not torch.isfinite(objective):
                    raise FloatingPointError("Non-finite multiclass objective.")
                optimizer.zero_grad(set_to_none=True)
                objective.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), CLASS_TRAINING_CONFIG["gradient_clip"]
                )
                optimizer.step()
                ce_sum += float(ce.detach().cpu()) * len(labels)
                n_rows += len(labels)

            validation_ce = _validation_ce(model, validation_tensor, n_classes)
            validation_cross, validation_pooled = 0.0, 0.0
            if constraint_loss_fn is not None:
                validation_cross, validation_pooled = _validation_mass_metrics(
                    model, prepared_validation, constraint_loss_fn
                )
            validation_total = validation_ce + float(lambda_norm) * validation_cross
            selection_active = scale_now >= 1.0
            if selection_active and scheduler is not None:
                scheduler.step(validation_total)

            history["train_ce"].append(ce_sum / max(1, n_rows))
            history["validation_ce"].append(validation_ce)
            history["validation_mass_cross"].append(validation_cross)
            history["validation_mass_pooled"].append(validation_pooled)
            history["validation_total"].append(validation_total)
            history["constraint_scale"].append(scale_now)
            history["learning_rate"].append(optimizer.param_groups[0]["lr"])

            if selection_active and validation_total < best_value - 1.0e-5:
                best_value = validation_total
                best_state = copy.deepcopy(model.state_dict())
                best_record = {
                    "epoch": epoch + 1,
                    "validation_ce": validation_ce,
                    "validation_mass_cross": validation_cross,
                    "validation_mass_pooled": validation_pooled,
                    "validation_total": validation_total,
                }
                stale = 0
            elif selection_active:
                stale += 1
            if epoch == 0 or (epoch + 1) % max(1, CLASS_EPOCHS // 7) == 0:
                print(
                    f"member {member}, epoch {epoch + 1:3d}: "
                    f"CE={validation_ce:.4f}, C_Z={validation_cross:.4g}, "
                    f"M_Z={validation_pooled:.4g}, "
                    f"ramp={scale_now:.2f}"
                )
            if stale >= int(CLASS_TRAINING_CONFIG["patience"]) and selection_active:
                break

        if best_state is None or best_record is None:
            raise RuntimeError("No finite classifier checkpoint was produced.")
        model.load_state_dict(best_state)
        model.eval()
        history["selected"] = [best_record]
        torch.save({
            "state_dict": model.state_dict(),
            "center": center,
            "scale": robust_scale,
            "history": history,
            "n_classes": n_classes,
            "input_dim": class_points.shape[-1],
            "model_config": CLASS_MODEL_CONFIG,
            "fingerprint": fingerprint,
        }, checkpoint)
        ensemble.append({
            "model": model,
            "center": center,
            "scale": robust_scale,
            "history": history,
            "checkpoint": checkpoint,
        })
        print(
            f"Selected member {member} epoch {best_record['epoch']}: "
            f"CE={best_record['validation_ce']:.4f}, "
            f"C_Z={best_record['validation_mass_cross']:.4g}, "
            f"M_Z={best_record['validation_mass_pooled']:.4g}"
        )
        print("Saved", checkpoint)
    return ensemble


@torch.no_grad()
def predict_shared_logits(ensemble, points, batch_size=65_536):
    points = _assert_finite("prediction points", points)
    original_shape = points.shape[:-1]
    flat = points.reshape(-1, points.shape[-1]).astype(np.float32)
    member_logits = []
    for pack in ensemble:
        transformed = _transform_classifier_points(flat, pack["center"], pack["scale"])
        chunks = []
        for start in range(0, len(flat), int(batch_size)):
            tensor = torch.as_tensor(
                transformed[start : start + int(batch_size)],
                dtype=torch.float32,
                device=device,
            )
            logits = pack["model"](tensor)
            logits = logits - logits.mean(dim=1, keepdim=True)
            chunks.append(logits.cpu().numpy())
        member_logits.append(np.concatenate(chunks, axis=0))
    averaged = np.mean(np.stack(member_logits), axis=0)
    return averaged.reshape(*original_shape, averaged.shape[-1])


@torch.no_grad()
def head_saturation_fraction(ensemble, points, head_name):
    points = _assert_finite("head diagnostic points", points)
    flat = points.reshape(-1, points.shape[-1]).astype(np.float32)
    saturated, count = 0, 0
    threshold = 0.8 * float(CLASS_MODEL_CONFIG["log_ratio_bound"])
    for pack in ensemble:
        transformed = _transform_classifier_points(flat, pack["center"], pack["scale"])
        for start in range(0, len(flat), 65_536):
            tensor = torch.as_tensor(
                transformed[start : start + 65_536],
                dtype=torch.float32,
                device=device,
            )
            raw = pack["model"].raw_ratio_heads(tensor)[head_name]
            saturated += int((raw.abs() > threshold).sum().cpu())
            count += int(raw.numel())
    return saturated / max(1, count)


def select_empirical_context_slices(context_pool):
    context_pool = _assert_finite("context pool", context_pool, ndim=2).astype(np.float32)
    center = np.median(context_pool, axis=0)
    mad = 1.4826 * np.median(np.abs(context_pool - center), axis=0)
    scale = np.where(mad > 1.0e-6, mad, context_pool.std(axis=0))
    scale = np.where(scale > 1.0e-6, scale, 1.0)
    score = np.max(np.abs((context_pool - center) / scale), axis=1)
    probabilities = np.array([0.50, 0.90, 0.97, 0.99, 0.995, 0.999, 0.9998, 1.0])
    targets = np.quantile(score, probabilities)
    indices = np.array([int(np.argmin(np.abs(score - value))) for value in targets])
    return context_pool[indices], score[indices], probabilities


def sample_only_flow_audit(flow_pack, contexts, truth_draws, scores, probabilities, name, seed):
    truth_draws = _assert_finite("simulator audit draws", truth_draws, ndim=3)
    flow_draws = _draw_conditional(flow_pack, contexts, truth_draws.shape[1], seed)
    rows = []
    quantiles = np.array([0.001, 0.01, 0.05, 0.50, 0.95, 0.99, 0.999])
    for index, (truth, learned) in enumerate(zip(truth_draws, flow_draws)):
        truth_scale = np.maximum(truth.std(axis=0), 1.0e-6)
        coordinate_w1 = np.array([
            wasserstein_distance(truth[:, feature], learned[:, feature])
            for feature in range(truth.shape[1])
        ]) / truth_scale
        mean_error = np.abs(learned.mean(axis=0) - truth.mean(axis=0)) / truth_scale
        quantile_error = np.max(np.abs(
            np.quantile(learned, quantiles, axis=0)
            - np.quantile(truth, quantiles, axis=0)
        ) / truth_scale, axis=0)

        projection_rng = np.random.default_rng(seed + 10_000 + index)
        directions = projection_rng.normal(size=(32, truth.shape[1]))
        directions /= np.linalg.norm(directions, axis=1, keepdims=True)
        truth_projection = truth @ directions.T
        learned_projection = learned @ directions.T
        projection_scale = np.maximum(truth_projection.std(axis=0), 1.0e-6)
        sliced_w1 = np.array([
            wasserstein_distance(truth_projection[:, direction], learned_projection[:, direction])
            for direction in range(len(directions))
        ]) / projection_scale

        truth_correlation = np.corrcoef(truth, rowvar=False)
        learned_correlation = np.corrcoef(learned, rowvar=False)
        correlation_error = np.max(np.abs(truth_correlation - learned_correlation))
        rows.append({
            "flow": name,
            "context quantile": float(probabilities[index]),
            "context score": float(scores[index]),
            "max coordinate W1 / truth sigma": float(np.max(coordinate_w1)),
            "q95 sliced W1 / truth sigma": float(np.quantile(sliced_w1, 0.95)),
            "max mean error / truth sigma": float(np.max(mean_error)),
            "max tail-quantile error / truth sigma": float(np.max(quantile_error)),
            "max correlation error": float(correlation_error),
        })
    result = pd.DataFrame(rows)
    if not np.isfinite(result.select_dtypes(include=[np.number])).all().all():
        raise FloatingPointError(f"{name}: non-finite sample-only flow audit.")
    print(
        f"{name} sample-only joint-tail audit: worst q95 sliced W1="
        f"{result['q95 sliced W1 / truth sigma'].max():.3f}."
    )
    return result


# Part I — nuisance marginalized implicitly

We first train two frozen proposal flows,

$$q_P(\mu\mid x),\qquad q_L^m(x\mid\mu),$$

from independent simulator samples.  Their simulations contain $\alpha\sim\rho_\alpha$, but $\alpha$ is omitted from both targets and contexts, so this is genuinely nuisance-marginal learning.

The three balanced classifier classes are

$$
\pi_S=\rho_\mu(\mu)p_m(x\mid\mu),\quad
\pi_P=m_\rho(x)q_P(\mu\mid x),\quad
\pi_L=\rho_\mu(\mu)q_L^m(x\mid\mu).
$$

Consequently $e^{s_S-s_P}$ and $e^{s_S-s_L}$ are the posterior and likelihood residuals.  Every group contains one simulator row, one posterior-reference row at the same $x$, and one likelihood-reference row at the same $\mu$.

### Proposal-flow numerical contracts

The stable scalar $q_P$ and $q_N$ proposals retain the audited one-layer monotone RQS and exact inverse/forward preflight introduced after the original float32 inverse-spline failure.  No sample is clipped, rejected, or redrawn.

The vector likelihood proposals now address a different failure exposed by the first full run.  A globally standardized target can legitimately lie far outside the old $[-5,5]$ spline interval, where linear-tail splines are identity maps.  We do **not** subtract the known Gaussian mean.  Instead, two full-support context-conditioned affine coupling layers are learned from the same samples before a wider 80-bin RQS stack.  This is an ordinary conditional normalizing flow with a fully accounted Jacobian, not an analytic-density shortcut.

After fitting, a sample-only audit compares fresh flow and simulator draws at empirical context quantiles through coordinate-wise Wasserstein, mean, and quantile errors.  Analytic likelihood values are not used.


In [ ]:
# Independent simulations for the two frozen proposals.
rng = np.random.default_rng(SEED + 10)
theta_qp = sample_design(N_FLOW, rng)
x_qp = simulate(theta_qp, rng)
set_torch_seed(SEED + 11)
q_p = train_spline_flow(
    theta_qp[:, :1],
    context=x_qp,
    checkpoint=MODEL_DIR / "q_p_mu_given_x_scalar_v2.pt",
    model_config=SCALAR_FLOW_MODEL_CONFIG,
    training_config=SCALAR_FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 11,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_qp, x_qp

rng = np.random.default_rng(SEED + 20)
theta_qlm = sample_vector_flow_design(N_FLOW, rng, score_columns=(0,))
# Only mu is a q_L^m context; alpha remains an ordinary design draw and
# is marginalized through simulation exactly as before.
theta_qlm[:, 1] = sample_alpha(N_FLOW, rng)
x_qlm = simulate(theta_qlm, rng)
set_torch_seed(SEED + 21)
q_lm = train_spline_flow(
    x_qlm,
    context=theta_qlm[:, :1],
    checkpoint=MODEL_DIR / "q_lm_x_given_mu_full_support_rqs_v2.pt",
    model_config=VECTOR_FLOW_MODEL_CONFIG,
    training_config=VECTOR_FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 21,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_qlm, x_qlm

# Sample-only q_L^m audit at central through extreme empirical mu ranks.
rng = np.random.default_rng(SEED + 22)
qlm_context_pool = sample_design(50_000 if not SMOKE_MODE else 2_000, rng)[:, :1]
qlm_audit_contexts, qlm_audit_scores, qlm_audit_probabilities = (
    select_empirical_context_slices(qlm_context_pool)
)
n_audit_contexts = len(qlm_audit_contexts)
mu_truth = np.repeat(qlm_audit_contexts[:, 0], N_FLOW_AUDIT_SAMPLES)
alpha_truth = sample_alpha(n_audit_contexts * N_FLOW_AUDIT_SAMPLES, rng)
qlm_truth_draws = simulate(
    np.column_stack([mu_truth, alpha_truth]), rng
).reshape(n_audit_contexts, N_FLOW_AUDIT_SAMPLES, 3)
qlm_tail_audit = sample_only_flow_audit(
    q_lm,
    qlm_audit_contexts,
    qlm_truth_draws,
    qlm_audit_scores,
    qlm_audit_probabilities,
    r"$q_L^m(x\mid\mu)$",
    SEED + 23,
)
display(qlm_tail_audit.style.format(precision=4))

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### Preflight: invertibility before building classifier classes

This audit uses independent ordinary and extreme defensive-design contexts.  It checks strict quantile monotonicity, inverse/forward closure, finite log density, exact repeated-seed sampling, and requires **zero** precision retries on this compact preflight.  The closure thresholds are deliberately loose relative to a healthy one-layer RQS ($q_{0.99}|z_{back}-z|\le10^{-4}$ and maximum $\le2\times10^{-3}$), so they catch a stiff or collapsed inverse rather than grade density accuracy.  A failed float64 retry stops here; it is never converted into rejection sampling.  This is numerical self-consistency—not a density-fidelity test; held-out closure, PIT, ESS, and Pareto-$k$ diagnostics come later.  The fallback remains available for a genuinely rare cancellation in the much larger production draw, where its kernel-call count is printed.


In [ ]:
rng = np.random.default_rng(SEED + 90)
theta_qp_audit = sample_design(16_384, rng)
theta_qp_tail = np.column_stack([
    np.array([-18, -15, -12, -10, 10, 12, 15, 18], dtype=np.float32),
    np.array([0, 6, -4, 3, -3, 4, -6, 0], dtype=np.float32),
])
x_qp_audit = np.concatenate([
    simulate(theta_qp_audit, rng),
    simulate(theta_qp_tail, rng),
])
audit_scalar_flow(q_p, x_qp_audit, r"$q_P(\mu\mid x)$", SEED + 91)
del theta_qp_audit, theta_qp_tail, x_qp_audit


In [ ]:
def build_three_class_groups(n_groups, q_p, q_lm, seed):
    rng = np.random.default_rng(seed)
    theta_s = sample_design(n_groups, rng)
    mu_s = theta_s[:, :1]
    x_s = simulate(theta_s, rng)
    mu_p = _draw_conditional(q_p, x_s, 1, seed + 1)[:, 0, :]
    x_l = _draw_conditional(q_lm, mu_s, 1, seed + 2)[:, 0, :]
    points_s = np.column_stack([mu_s, x_s])
    points_p = np.column_stack([mu_p, x_s])
    points_l = np.column_stack([mu_s, x_l])
    groups = np.stack([points_s, points_p, points_l], axis=1).astype(np.float32)
    return _assert_finite("three-class groups", groups, ndim=3)

three_retry_before = _rqs_retry_count()
three_class_groups = build_three_class_groups(
    N_CLASS, q_p, q_lm, SEED + 100
)
print("Three-class grouped tensor:", three_class_groups.shape)
print(
    "Inverse-RQS kernel calls retried in float64 during three-class construction:",
    _rqs_retry_count() - three_retry_before,
)


## Cross-fitted conditional normalization

The residual ratios must have unit conditional mass:

$$
Z_P(x)=\mathbb E_{q_P}[e^{s_S-s_P}],\qquad
Z_L(\mu)=\mathbb E_{q_L^m}[e^{s_S-s_L}].
$$

The previous squared $\log\widehat Z$ loss was finite-Monte-Carlo biased: even when $Z=1$, $\mathbb E[(\log\widehat Z)^2]>0$ for a nonconstant ratio, so it can reward shrinking a genuine correction toward one.

For every conditioning anchor we now draw two independent proposal sets $A$ and $B$ and minimize

$$
\widehat{\mathcal L}_{Z,\mathrm{cross}}
=(\widehat Z_A-1)(\widehat Z_B-1).
$$

Conditional independence gives

$$
\mathbb E[\widehat{\mathcal L}_{Z,\mathrm{cross}}\mid c]=(Z(c)-1)^2.
$$

A minibatch estimate may be negative; its expectation is non-negative.  Checkpoint selection averages the same signed cross moment over several large, disjoint held-out banks.  The positive pooled estimate $[(\widehat Z_A+\widehat Z_B)/2-1]^2$ is reported as a variance-sensitive diagnostic only.  Eight independent full-run training banks rotate by epoch, the validation banks are disjoint, and 25% of their anchors are selected from empirical design tails.  Fresh post-training banks provide the final closure test.  Everything uses proposal and simulator samples only.


In [ ]:
def build_three_normalization_bundle(n_groups, n_inner, q_p, q_lm, seed):
    rng = np.random.default_rng(seed)

    theta_x = sample_tail_enriched_design(n_groups, rng)
    x_zp = simulate(theta_x, rng)
    mu_zp_a = _draw_conditional(q_p, x_zp, n_inner, seed + 1)
    mu_zp_b = _draw_conditional(q_p, x_zp, n_inner, seed + 2)
    x_zp_repeat = np.repeat(x_zp[:, None, :], n_inner, axis=1)

    mu_zl = sample_tail_enriched_design(n_groups, rng)[:, :1]
    x_zl_a = _draw_conditional(q_lm, mu_zl, n_inner, seed + 3)
    x_zl_b = _draw_conditional(q_lm, mu_zl, n_inner, seed + 4)
    mu_zl_repeat = np.repeat(mu_zl[:, None, :], n_inner, axis=1)

    bundle = {
        "zp_a_points": np.concatenate([mu_zp_a, x_zp_repeat], axis=2),
        "zp_b_points": np.concatenate([mu_zp_b, x_zp_repeat], axis=2),
        "zl_a_points": np.concatenate([mu_zl_repeat, x_zl_a], axis=2),
        "zl_b_points": np.concatenate([mu_zl_repeat, x_zl_b], axis=2),
    }
    for name, value in bundle.items():
        _assert_finite(name, value)
        if len(value) != n_groups:
            raise RuntimeError(f"{name} lost its anchor-group axis.")
    return bundle


def three_normalization_loss(model, bundle, index):
    logits_zp_a = _constraint_logits(model, bundle, "zp_a_points", index, 3)
    logits_zp_b = _constraint_logits(model, bundle, "zp_b_points", index, 3)
    cross_zp, monitor_zp = _partition_cross_mass(logits_zp_a, logits_zp_b, 0, 1)

    logits_zl_a = _constraint_logits(model, bundle, "zl_a_points", index, 3)
    logits_zl_b = _constraint_logits(model, bundle, "zl_b_points", index, 3)
    cross_zl, monitor_zl = _partition_cross_mass(logits_zl_a, logits_zl_b, 0, 2)
    return 0.5 * (cross_zp + cross_zl), 0.5 * (monitor_zp + monitor_zl)


constraints_three_train = [
    build_three_normalization_bundle(
        N_NORM_TRAIN_GROUPS,
        N_NORM_TRAIN_INNER,
        q_p,
        q_lm,
        SEED + 200 + 20 * bank,
    )
    for bank in range(N_NORM_TRAIN_BANKS)
]
constraints_three_validation = [
    build_three_normalization_bundle(
        N_NORM_VALID_GROUPS,
        N_NORM_VALID_INNER,
        q_p,
        q_lm,
        SEED + 300 + 20 * bank,
    )
    for bank in range(N_NORM_VALID_BANKS)
]


## Controlled normalization ablation

Both ensembles see exactly the same class triplets, group split, structured ratio architecture, initial seeds, optimizer reset, learning-rate schedule, and rotating normalization banks.  The first has $\lambda_Z=0$ and is selected by CE; the second uses $\lambda_Z=0.20$ and is selected by CE plus the cross-fitted held-out mass moment.

There is no bridge coefficient or bridge loss.  Posterior/likelihood agreement and evidence invariance remain outcomes to be checked rather than quantities optimized directly.


In [ ]:
three_ce_only = train_multiclass_ensemble(
    three_class_groups,
    n_classes=3,
    checkpoint_dir=MODEL_DIR / "three_class_ce_only",
    seed_base=SEED + 400,
    constraint_train=constraints_three_train,
    constraint_validation=constraints_three_validation,
    constraint_loss_fn=three_normalization_loss,
    lambda_norm=0.0,
)

three_normalized = train_multiclass_ensemble(
    three_class_groups,
    n_classes=3,
    checkpoint_dir=MODEL_DIR / "three_class_cross_normalized",
    seed_base=SEED + 400,
    constraint_train=constraints_three_train,
    constraint_validation=constraints_three_validation,
    constraint_loss_fn=three_normalization_loss,
    lambda_norm=LAMBDA_NORM_3,
)


## Independent closure of the three-class experiment

Fresh proposal draws are used throughout this section.  We compare

- the posterior route $q_Pe^{s_S-s_P}/Z_P$;
- the likelihood route $\rho_\mu q_L^me^{s_S-s_L}/Z_L$;
- the implied evidence consistency as a **held-out diagnostic**.

The highlighted consistency region is where the analytic validation posterior is at least $10^{-4}$ of its peak.  The analytic likelihood is used only here, after training.  Large off-support excursions remain in tabulated maxima but do not dominate the supported-region curve.


In [ ]:
def three_log_zp(ensemble, x_values, n_reference, seed):
    x_values = np.atleast_2d(np.asarray(x_values, dtype=np.float32))
    mu = _draw_conditional(q_p, x_values, n_reference, seed)
    points = np.concatenate(
        [mu, np.repeat(x_values[:, None, :], n_reference, axis=1)], axis=2
    )
    logits = predict_shared_logits(ensemble, points)
    return logsumexp(logits[..., 0] - logits[..., 1], axis=1) - np.log(n_reference)

def three_log_zl(ensemble, mu_values, n_reference, seed):
    mu_values = np.asarray(mu_values, dtype=np.float32).reshape(-1, 1)
    x = _draw_conditional(q_lm, mu_values, n_reference, seed)
    points = np.concatenate(
        [np.repeat(mu_values[:, None, :], n_reference, axis=1), x], axis=2
    )
    logits = predict_shared_logits(ensemble, points)
    return logsumexp(logits[..., 0] - logits[..., 2], axis=1) - np.log(n_reference)

def three_closure(ensemble, mu_grid, x_observed, n_reference, seed):
    mu_grid = np.asarray(mu_grid, dtype=float)
    x_grid = np.repeat(np.asarray(x_observed)[None, :], len(mu_grid), axis=0)
    points = np.column_stack([mu_grid, x_grid]).astype(np.float32)
    logits = predict_shared_logits(ensemble, points)
    log_qp = spline_flow_log_prob(q_p, mu_grid[:, None], context=x_grid)
    log_ql = spline_flow_log_prob(q_lm, x_grid, context=mu_grid[:, None])
    log_zp = float(three_log_zp(ensemble, x_observed, n_reference, seed)[0])
    log_zl = three_log_zl(ensemble, mu_grid, n_reference, seed + 1)

    log_posterior_route = log_qp + logits[:, 0] - logits[:, 1] - log_zp
    posterior_route, _ = normalize_log_curve(log_posterior_route, mu_grid)
    log_likelihood_route = log_ql + logits[:, 0] - logits[:, 2] - log_zl
    likelihood_posterior, _ = normalize_log_curve(
        design_mu_logpdf(mu_grid) + log_likelihood_route, mu_grid
    )
    log_evidence_consistency = (
        design_mu_logpdf(mu_grid)
        + log_ql
        - log_qp
        + logits[:, 1]
        - logits[:, 2]
        + log_zp
        - log_zl
    )
    return {
        "posterior": posterior_route,
        "likelihood_posterior": likelihood_posterior,
        "log_evidence_consistency": log_evidence_consistency,
        "log_zp": log_zp,
        "log_zl": log_zl,
        "log_ratio_grid": logits[:, 0] - logits[:, 1],
    }

MU_GRID = np.linspace(-3.8, 3.8, 321)
truth_mu, _ = normalize_log_curve(
    design_mu_logpdf(MU_GRID) + marginal_log_likelihood(X_OBS, MU_GRID),
    MU_GRID,
)
MU_EVIDENCE_GRID = np.linspace(-10.0, 10.0, 4001)
_, LOG_EVIDENCE_TRUTH = normalize_log_curve(
    design_mu_logpdf(MU_EVIDENCE_GRID)
    + marginal_log_likelihood(X_OBS, MU_EVIDENCE_GRID),
    MU_EVIDENCE_GRID,
)
q_p_curve, _ = normalize_log_curve(
    spline_flow_log_prob(
        q_p,
        MU_GRID[:, None],
        context=np.repeat(X_OBS[None, :], len(MU_GRID), axis=0),
    ),
    MU_GRID,
)
closure_ce_only = three_closure(
    three_ce_only, MU_GRID, X_OBS, N_DIAGNOSTIC_REFERENCE, SEED + 500
)
closure_normalized = three_closure(
    three_normalized, MU_GRID, X_OBS, N_DIAGNOSTIC_REFERENCE, SEED + 500
)

# Held-out conditional normalizers at many independent contexts.
rng = np.random.default_rng(SEED + 510)
n_context_check = 36 if SMOKE_MODE else (80 if FAST_MODE else 160)
theta_check = sample_design(n_context_check, rng)
x_check = simulate(theta_check, rng)
mu_check = sample_mu(n_context_check, rng)
heldout_norm = {}
for name, ensemble in [("CE only", three_ce_only), ("CE + normalization", three_normalized)]:
    heldout_norm[name] = {
        "log_zp": three_log_zp(
            ensemble, x_check, N_DIAGNOSTIC_REFERENCE, SEED + 520
        ),
        "log_zl": three_log_zl(
            ensemble, mu_check, N_DIAGNOSTIC_REFERENCE, SEED + 521
        ),
    }

# Importance-tail diagnostics at x_obs.
mu_tail = _draw_conditional(
    q_p, X_OBS[None, :],
    2_000 if SMOKE_MODE else (20_000 if FAST_MODE else 80_000),
    SEED + 530,
)[0]
x_tail = np.repeat(X_OBS[None, :], len(mu_tail), axis=0)
tail_points = np.column_stack([mu_tail, x_tail])
tail_summaries = {}
tail_log_weights = {}
for name, ensemble in [("CE only", three_ce_only), ("CE + normalization", three_normalized)]:
    logits = predict_shared_logits(ensemble, tail_points)
    log_weights = logits[:, 0] - logits[:, 1]
    tail_log_weights[name] = log_weights
    tail_summaries[name] = importance_tail_summary(log_weights)

mu_likelihood_tail = float(MU_GRID[np.argmax(truth_mu)])
x_likelihood_tail = _draw_conditional(
    q_lm,
    [[mu_likelihood_tail]],
    len(mu_tail),
    SEED + 531,
)[0]
likelihood_tail_points = np.column_stack([
    np.full(len(x_likelihood_tail), mu_likelihood_tail),
    x_likelihood_tail,
])
likelihood_tail_summaries = {}
for name, ensemble in [("CE only", three_ce_only), ("CE + normalization", three_normalized)]:
    logits = predict_shared_logits(ensemble, likelihood_tail_points)
    likelihood_tail_summaries[name] = importance_tail_summary(
        logits[:, 0] - logits[:, 2]
    )

rows = []
for name, closure, ensemble in [
    ("CE only", closure_ce_only, three_ce_only),
    ("CE + normalization", closure_normalized, three_normalized),
]:
    normalizers = heldout_norm[name]
    support_mask = truth_mu > 1.0e-4 * truth_mu.max()
    consistency_delta = closure["log_evidence_consistency"] - LOG_EVIDENCE_TRUTH
    tails = tail_summaries[name]
    all_log_z = np.concatenate([normalizers["log_zp"], normalizers["log_zl"]])
    rows.append({
        "objective": name,
        "posterior IAE": integrated_absolute_error(truth_mu, closure["posterior"], MU_GRID),
        "likelihood-route IAE": integrated_absolute_error(
            truth_mu, closure["likelihood_posterior"], MU_GRID
        ),
        "evidence RMS (supported)": float(np.sqrt(np.mean(consistency_delta[support_mask] ** 2))),
        "RMS log Z": float(np.sqrt(np.mean(all_log_z**2))),
        "q95 |log Z|": float(np.quantile(np.abs(all_log_z), 0.95)),
        "max |log Z|": float(np.max(np.abs(all_log_z))),
        "a-head saturation (q_P)": head_saturation_fraction(
            ensemble, tail_points, "a"
        ),
        "c-head saturation (q_Lm)": head_saturation_fraction(
            ensemble, likelihood_tail_points, "c"
        ),
        "ESS fraction": tails["ESS_fraction"],
        "Pareto k": tails["pareto_k"],
        "max weight": tails["max_weight_fraction"],
        "likelihood ESS fraction": likelihood_tail_summaries[name]["ESS_fraction"],
        "likelihood Pareto k": likelihood_tail_summaries[name]["pareto_k"],
        "likelihood max weight": likelihood_tail_summaries[name]["max_weight_fraction"],
    })
three_summary = pd.DataFrame(rows)
display(three_summary.style.format(precision=4))


In [ ]:
CE_ONLY_COLOR = "#D55E00"
FULL3_COLOR = "#0072B2"
FULL4_COLOR = "#009E73"

fig, axes = plt.subplots(2, 2, figsize=(12.2, 8.8), constrained_layout=True)

axes[0, 0].plot(MU_GRID, truth_mu, color="black", lw=2.4, label="analytic truth")
axes[0, 0].plot(MU_GRID, q_p_curve, color="0.6", lw=1.5, ls=":", label=r"proposal $q_P$")
axes[0, 0].plot(
    MU_GRID, closure_ce_only["posterior"], color=CE_ONLY_COLOR, lw=2.0,
    label="structured CE only",
)
axes[0, 0].plot(
    MU_GRID, closure_normalized["posterior"], color=FULL3_COLOR, lw=2.2,
    label="CE + normalization: posterior",
)
axes[0, 0].plot(
    MU_GRID, closure_normalized["likelihood_posterior"], color=FULL3_COLOR,
    lw=1.8, ls="--", label="CE + normalization: likelihood",
)
axes[0, 0].set(xlabel=r"$\mu$", ylabel="posterior density", title="(a) Posterior closure")
axes[0, 0].legend(fontsize=8.5)

supported = truth_mu > 1.0e-4 * truth_mu.max()
for label, closure, color in [
    ("CE only", closure_ce_only, CE_ONLY_COLOR),
    ("CE + normalization", closure_normalized, FULL3_COLOR),
]:
    residual = closure["log_evidence_consistency"] - LOG_EVIDENCE_TRUTH
    axes[0, 1].plot(
        MU_GRID, np.where(supported, residual, np.nan),
        color=color, lw=2, label=label,
    )
axes[0, 1].axhline(0.0, color="black", lw=1)
axes[0, 1].set(
    xlabel=r"$\mu$", ylabel=r"$\log\widehat m-\log m_{\rm true}$",
    title="(b) Held-out evidence consistency\n(posterior-supported region)",
)
axes[0, 1].legend(fontsize=8.5)

jitter = {"CE only": -0.06, "CE + normalization": 0.06}
color_map = {"CE only": CE_ONLY_COLOR, "CE + normalization": FULL3_COLOR}
for name in ["CE only", "CE + normalization"]:
    values_p = heldout_norm[name]["log_zp"]
    values_l = heldout_norm[name]["log_zl"]
    rng_plot = np.random.default_rng(SEED + (1 if name == "CE only" else 2))
    axes[1, 0].scatter(
        np.full_like(values_p, 0.0 + jitter[name]) + rng_plot.normal(0, 0.012, len(values_p)),
        values_p, s=12, alpha=0.45, color=color_map[name],
    )
    axes[1, 0].scatter(
        np.full_like(values_l, 1.0 + jitter[name]) + rng_plot.normal(0, 0.012, len(values_l)),
        values_l, s=12, alpha=0.45, color=color_map[name], label=name,
    )
axes[1, 0].axhline(0.0, color="black", lw=1)
axes[1, 0].set(
    xticks=[0, 1], xticklabels=[r"$\log Z_P(x)$", r"$\log Z_L(\mu)$"],
    ylabel="held-out conditional log normalizer",
    title="(c) Independent normalization closure",
)
axes[1, 0].legend(fontsize=8.5)

for name, color in [("CE only", CE_ONLY_COLOR), ("CE + normalization", FULL3_COLOR)]:
    log_w = tail_log_weights[name]
    weights = np.exp(log_w - logsumexp(log_w)) * len(log_w)
    ordered = np.sort(np.maximum(weights, np.finfo(float).tiny))
    survival = 1.0 - np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
    axes[1, 1].plot(ordered, survival, color=color, lw=1.8, label=name)
axes[1, 1].set(
    xscale="log", yscale="log", xlabel=r"normalized correction $r_P/\bar{r_P}$",
    ylabel="empirical survival", title="(d) Posterior-correction tail",
)
axes[1, 1].legend(fontsize=8.5)
for ax in axes.flat:
    ax.grid(alpha=0.25)

export_exercise9_multiclass_figure(fig, "three_class_ce_vs_normalized")
plt.show()


**Figure 1 interpretation.**  The two classifiers share proposals, triplets, group split, structured architecture, initialization, optimizer schedule, and candidate normalization banks; only $\lambda_Z$ changes from zero to its configured value.  Panel (b) is an independently normalized evidence-consistency diagnostic and never enters training.  Panel (c) uses fresh proposal draws and reports actual conditional mass closure.  The maximum column in the table should be read together with q95 so a single broad-design tail does not masquerade as a bulk normalization failure.


## Held-out simulation-based calibration of the normalization ablation

Closure at one $x_{\rm obs}$ is not evidence of amortized calibration.  We therefore draw a new design sample $(\mu_i,\alpha_i,x_i)$ and compute posterior PIT values from $q_P$ and both residual classifiers.  A calibrated design posterior has uniform PIT values and nominal equal-tailed coverage.

The full mode now uses 2,000 contexts, making this comparison substantially more discriminating than the earlier 200-context diagnostic.  A paper result should still repeat the complete training over independent seeds and show paired uncertainty bands.


In [ ]:
rng = np.random.default_rng(SEED + 1300)
theta_calibration = sample_design(N_CALIBRATION_CONTEXTS, rng)
x_calibration = simulate(theta_calibration, rng)
mu_calibration = _draw_conditional(
    q_p, x_calibration, N_CALIBRATION_SAMPLES, SEED + 1301
)[..., 0]
calibration_points = np.concatenate([
    mu_calibration[..., None],
    np.repeat(x_calibration[:, None, :], N_CALIBRATION_SAMPLES, axis=1),
], axis=2)
below_truth = mu_calibration <= theta_calibration[:, 0, None]

pit_values = {"proposal q_P": below_truth.mean(axis=1)}
for name, ensemble in [
    ("CE only", three_ce_only),
    ("CE + normalization", three_normalized),
]:
    logits = predict_shared_logits(ensemble, calibration_points)
    log_weights = logits[..., 0] - logits[..., 1]
    weights = np.exp(log_weights - logsumexp(log_weights, axis=1, keepdims=True))
    pit_values[name] = np.sum(weights * below_truth, axis=1)

NOMINAL_COVERAGE = np.linspace(0.05, 0.95, 19)
coverage_values = {
    name: np.array([
        np.mean((pit >= 0.5 * (1.0 - level)) & (pit <= 0.5 * (1.0 + level)))
        for level in NOMINAL_COVERAGE
    ])
    for name, pit in pit_values.items()
}
calibration_rows = []
for name, pit in pit_values.items():
    ordered = np.sort(pit)
    uniform_quantiles = (np.arange(len(ordered)) + 0.5) / len(ordered)
    calibration_rows.append({
        "method": name,
        "PIT KS distance": float(np.max(np.abs(ordered - uniform_quantiles))),
        "max coverage error": float(np.max(np.abs(coverage_values[name] - NOMINAL_COVERAGE))),
    })
calibration_summary = pd.DataFrame(calibration_rows)
display(calibration_summary.style.format(precision=4))

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.3), constrained_layout=True)
calibration_colors = {
    "proposal q_P": "0.55",
    "CE only": CE_ONLY_COLOR,
    "CE + normalization": FULL3_COLOR,
}
for name, pit in pit_values.items():
    ordered = np.sort(pit)
    empirical_cdf = np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
    axes[0].plot(ordered, empirical_cdf, lw=1.9, color=calibration_colors[name], label=name)
    axes[1].plot(
        NOMINAL_COVERAGE, coverage_values[name], lw=1.9,
        color=calibration_colors[name], label=name,
    )
axes[0].plot([0, 1], [0, 1], color="black", ls="--", lw=1)
axes[0].set(
    xlabel="posterior PIT", ylabel="empirical CDF",
    title="(a) Held-out posterior PIT",
)
axes[1].plot([0, 1], [0, 1], color="black", ls="--", lw=1)
binomial_sigma = np.sqrt(
    NOMINAL_COVERAGE * (1.0 - NOMINAL_COVERAGE) / N_CALIBRATION_CONTEXTS
)
axes[1].fill_between(
    NOMINAL_COVERAGE,
    np.maximum(0.0, NOMINAL_COVERAGE - binomial_sigma),
    np.minimum(1.0, NOMINAL_COVERAGE + binomial_sigma),
    color="0.7", alpha=0.2, linewidth=0, label=r"$\pm1\sigma$ binomial",
)
axes[1].set(
    xlabel="nominal equal-tailed coverage", ylabel="empirical coverage",
    title="(b) Held-out coverage",
)
for ax in axes:
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
export_exercise9_multiclass_figure(fig, "three_class_heldout_calibration")
plt.show()


# Part II — the nuisance parameter is explicit

We now train

$$q_N(\alpha\mid x,\mu),\qquad q_L(x\mid\mu,\alpha),$$

while reusing the frozen marginal $q_P(\mu\mid x)$.  The four balanced classes are

$$
\begin{aligned}
\Pi_S &= \rho_\mu\rho_\alpha p(x\mid\mu,\alpha),\\
\Pi_N &= \rho_\mu p_m(x\mid\mu)q_N(\alpha\mid x,\mu),\\
\Pi_P &= m_\rho(x)q_P(\mu\mid x)q_N(\alpha\mid x,\mu),\\
\Pi_L &= \rho_\mu\rho_\alpha q_L(x\mid\mu,\alpha).
\end{aligned}
$$

The ratio-head architecture supplies $r_P=e^a$, $r_N=e^b$, $r_L=e^c$, and $r_{PN}=e^{a+b}$.  Only $a$ excludes $\alpha$: this encodes the exact cancellation of the shared $q_N$ factor between classes $N$ and $P$.  The network is not given $p$, $q$, $\rho$, an analytic mean, or any log density as an input.


In [ ]:
# q_L^m is no longer needed for nuisance-aware training.  Keep its pack
# usable for rerunning Part I, but release GPU memory.
q_lm["flow"].to(torch.device("cpu"))
if torch.cuda.is_available():
    torch.cuda.empty_cache()

rng = np.random.default_rng(SEED + 600)
theta_qn = sample_design(N_FLOW, rng)
x_qn = simulate(theta_qn, rng)
set_torch_seed(SEED + 601)
q_n = train_spline_flow(
    theta_qn[:, 1:2],
    context=np.column_stack([x_qn, theta_qn[:, 0]]),
    checkpoint=MODEL_DIR / "q_n_alpha_given_x_mu_scalar_v2.pt",
    model_config=SCALAR_FLOW_MODEL_CONFIG,
    training_config=SCALAR_FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 601,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_qn, x_qn

rng = np.random.default_rng(SEED + 610)
theta_ql = sample_vector_flow_design(N_FLOW, rng, score_columns=(0, 1))
x_ql = simulate(theta_ql, rng)
set_torch_seed(SEED + 611)
q_l = train_spline_flow(
    x_ql,
    context=theta_ql,
    checkpoint=MODEL_DIR / "q_l_x_given_mu_alpha_full_support_rqs_v2.pt",
    model_config=VECTOR_FLOW_MODEL_CONFIG,
    training_config=VECTOR_FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 611,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_ql, x_ql

# The same sample-only tail audit, now at explicit (mu,alpha) contexts.
rng = np.random.default_rng(SEED + 612)
ql_context_pool = sample_design(50_000 if not SMOKE_MODE else 2_000, rng)
ql_audit_contexts, ql_audit_scores, ql_audit_probabilities = (
    select_empirical_context_slices(ql_context_pool)
)
n_audit_contexts = len(ql_audit_contexts)
ql_truth_draws = simulate(
    np.repeat(ql_audit_contexts, N_FLOW_AUDIT_SAMPLES, axis=0), rng
).reshape(n_audit_contexts, N_FLOW_AUDIT_SAMPLES, 3)
ql_tail_audit = sample_only_flow_audit(
    q_l,
    ql_audit_contexts,
    ql_truth_draws,
    ql_audit_scores,
    ql_audit_probabilities,
    r"$q_L(x\mid\mu,\alpha)$",
    SEED + 613,
)
display(ql_tail_audit.style.format(precision=4))

fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.3), constrained_layout=True)
for table, color, label in [
    (qlm_tail_audit, FULL3_COLOR, r"$q_L^m$"),
    (ql_tail_audit, FULL4_COLOR, r"$q_L$"),
]:
    axes[0].plot(
        table["context score"], table["q95 sliced W1 / truth sigma"],
        marker="o", lw=1.8, color=color, label=label,
    )
    axes[1].plot(
        table["context score"], table["max tail-quantile error / truth sigma"],
        marker="o", lw=1.8, color=color, label=label,
    )
axes[0].set(
    xlabel="empirical context-tail score", ylabel="q95 sliced W1 / truth sigma",
    title="(a) Joint sample-only transport",
)
axes[1].set(
    xlabel="empirical context-tail score", ylabel="max tail-quantile error / truth sigma",
    title="(b) 0.1% through 99.9% quantiles",
)
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8.5)
export_exercise9_multiclass_figure(fig, "vector_flow_sample_only_tail_audit")
plt.show()

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### Preflight for the explicit nuisance proposal

The identical scalar-flow contract is now applied to $q_N(\alpha\mid x,\mu)$ before any four-class rows or nested posterior references are generated.


In [ ]:
rng = np.random.default_rng(SEED + 618)
theta_qn_audit = sample_design(16_384, rng)
theta_qn_tail = np.column_stack([
    np.array([-10, -8, -5, 0, 0, 5, 8, 10], dtype=np.float32),
    np.array([-14, 12, -10, -8, 8, 10, -12, 14], dtype=np.float32),
])
theta_qn_all = np.concatenate([theta_qn_audit, theta_qn_tail])
x_qn_audit = simulate(theta_qn_all, rng)
qn_audit_context = np.column_stack([x_qn_audit, theta_qn_all[:, 0]])
audit_scalar_flow(q_n, qn_audit_context, r"$q_N(\alpha\mid x,\mu)$", SEED + 619)
del theta_qn_audit, theta_qn_tail, theta_qn_all, x_qn_audit, qn_audit_context


In [ ]:
def _qn_context(x, mu):
    x = np.atleast_2d(np.asarray(x, dtype=np.float32))
    mu = np.asarray(mu, dtype=np.float32).reshape(-1, 1)
    if len(x) == 1 and len(mu) > 1:
        x = np.repeat(x, len(mu), axis=0)
    if len(x) != len(mu):
        raise ValueError("x and mu rows must match.")
    return np.column_stack([x, mu])

def _draw_joint_reference(x, n_samples, seed):
    x = np.atleast_2d(np.asarray(x, dtype=np.float32))
    mu = _draw_conditional(q_p, x, n_samples, seed)
    x_repeat = np.repeat(x[:, None, :], n_samples, axis=1)
    flat_context = _qn_context(x_repeat.reshape(-1, 3), mu.reshape(-1, 1))
    alpha = _draw_conditional(q_n, flat_context, 1, seed + 1)[:, 0, :]
    alpha = alpha.reshape(len(x), n_samples, 1)
    return mu, alpha

def build_four_class_groups(n_groups, q_p, q_n, q_l, seed):
    rng = np.random.default_rng(seed)
    theta_s = sample_design(n_groups, rng)
    mu_s, alpha_s = theta_s[:, :1], theta_s[:, 1:2]
    x_s = simulate(theta_s, rng)

    alpha_n = _draw_conditional(q_n, _qn_context(x_s, mu_s), 1, seed + 1)[:, 0, :]
    mu_p = _draw_conditional(q_p, x_s, 1, seed + 2)[:, 0, :]
    alpha_p = _draw_conditional(q_n, _qn_context(x_s, mu_p), 1, seed + 3)[:, 0, :]
    x_l = _draw_conditional(q_l, theta_s, 1, seed + 4)[:, 0, :]

    points_s = np.column_stack([mu_s, alpha_s, x_s])
    points_n = np.column_stack([mu_s, alpha_n, x_s])
    points_p = np.column_stack([mu_p, alpha_p, x_s])
    points_l = np.column_stack([mu_s, alpha_s, x_l])
    groups = np.stack([points_s, points_n, points_p, points_l], axis=1).astype(
        np.float32
    )
    return _assert_finite("four-class groups", groups, ndim=3)

four_retry_before = _rqs_retry_count()
four_class_groups = build_four_class_groups(
    N_CLASS, q_p, q_n, q_l, SEED + 620
)
print("Four-class grouped tensor:", four_class_groups.shape)
print(
    "Inverse-RQS kernel calls retried in float64 during four-class construction:",
    _rqs_retry_count() - four_retry_before,
)


## Nuisance-aware structured ratios and normalization

The four conditional residuals are

$$
r_P=e^{s_N-s_P}=e^a,\qquad
r_N=e^{s_S-s_N}=e^b,\qquad
r_L=e^{s_S-s_L}=e^c,\qquad
r_{PN}=e^{s_S-s_P}=e^{a+b}.
$$

Because $a(\mu,x)$ excludes $\alpha$ by architecture, the three head-local normalization conditions are sufficient:

$$
Z_P(x)=\mathbb E_{q_P}[e^a],\qquad
Z_N(x,\mu)=\mathbb E_{q_N}[e^b],\qquad
Z_L(\mu,\alpha)=\mathbb E_{q_L}[e^c].
$$

$Z_{PN}$ is deliberately not a fourth training penalty: if the first two conditional moments equal one, iterated expectation gives $Z_{PN}=1$.  We nevertheless estimate it independently after training as a composition diagnostic.  Each trained partition again uses independent A/B samples and the unbiased cross-mass estimator.  No nuisance-marginal or evidence consistency variance appears in the objective.


In [ ]:
def build_four_normalization_bundle(n_groups, n_inner, q_p, q_n, q_l, seed):
    rng = np.random.default_rng(seed)

    # Z_P(x): a(mu,x) does not depend on alpha, so q_N draws are unnecessary.
    theta_zp_anchor = sample_tail_enriched_design(n_groups, rng)
    x_zp = simulate(theta_zp_anchor, rng)
    mu_zp_a = _draw_conditional(q_p, x_zp, n_inner, seed + 1)
    mu_zp_b = _draw_conditional(q_p, x_zp, n_inner, seed + 2)
    x_zp_repeat = np.repeat(x_zp[:, None, :], n_inner, axis=1)
    alpha_filler = np.zeros_like(mu_zp_a)

    # Z_N(x,mu): independent q_N halves at fixed sampled (x,mu).
    theta_zn_anchor = sample_tail_enriched_design(n_groups, rng)
    x_zn = simulate(theta_zn_anchor, rng)
    mu_zn = theta_zn_anchor[:, :1]
    qn_context = _qn_context(x_zn, mu_zn)
    alpha_zn_a = _draw_conditional(q_n, qn_context, n_inner, seed + 3)
    alpha_zn_b = _draw_conditional(q_n, qn_context, n_inner, seed + 4)
    mu_zn_repeat = np.repeat(mu_zn[:, None, :], n_inner, axis=1)
    x_zn_repeat = np.repeat(x_zn[:, None, :], n_inner, axis=1)

    # Z_L(theta): independent q_L halves at fixed sampled theta.
    theta_zl = sample_tail_enriched_design(n_groups, rng)
    x_zl_a = _draw_conditional(q_l, theta_zl, n_inner, seed + 5)
    x_zl_b = _draw_conditional(q_l, theta_zl, n_inner, seed + 6)
    theta_zl_repeat = np.repeat(theta_zl[:, None, :], n_inner, axis=1)

    bundle = {
        "zp_a_points": np.concatenate([mu_zp_a, alpha_filler, x_zp_repeat], axis=2),
        "zp_b_points": np.concatenate([mu_zp_b, alpha_filler, x_zp_repeat], axis=2),
        "zn_a_points": np.concatenate([mu_zn_repeat, alpha_zn_a, x_zn_repeat], axis=2),
        "zn_b_points": np.concatenate([mu_zn_repeat, alpha_zn_b, x_zn_repeat], axis=2),
        "zl_a_points": np.concatenate([theta_zl_repeat, x_zl_a], axis=2),
        "zl_b_points": np.concatenate([theta_zl_repeat, x_zl_b], axis=2),
    }
    for name, value in bundle.items():
        _assert_finite(name, value)
        if len(value) != n_groups:
            raise RuntimeError(f"{name} lost its anchor-group axis.")
    return bundle


def four_normalization_loss(model, bundle, index):
    pairs = [
        ("zp", 1, 2),
        ("zn", 0, 1),
        ("zl", 0, 3),
    ]
    cross_terms, monitor_terms = [], []
    for prefix, numerator, denominator in pairs:
        logits_a = _constraint_logits(model, bundle, f"{prefix}_a_points", index, 4)
        logits_b = _constraint_logits(model, bundle, f"{prefix}_b_points", index, 4)
        cross, monitor = _partition_cross_mass(
            logits_a, logits_b, numerator, denominator
        )
        cross_terms.append(cross)
        monitor_terms.append(monitor)
    return torch.stack(cross_terms).mean(), torch.stack(monitor_terms).mean()


constraints_four_train = [
    build_four_normalization_bundle(
        N_NORM_TRAIN_GROUPS,
        N_NORM_TRAIN_INNER,
        q_p,
        q_n,
        q_l,
        SEED + 700 + 20 * bank,
    )
    for bank in range(N_NORM_TRAIN_BANKS)
]
constraints_four_validation = [
    build_four_normalization_bundle(
        N_NORM_VALID_GROUPS,
        N_NORM_VALID_INNER,
        q_p,
        q_n,
        q_l,
        SEED + 800 + 20 * bank,
    )
    for bank in range(N_NORM_VALID_BANKS)
]


In [ ]:
four_normalized = train_multiclass_ensemble(
    four_class_groups,
    n_classes=4,
    checkpoint_dir=MODEL_DIR / "four_class_structured_cross_normalized",
    seed_base=SEED + 900,
    constraint_train=constraints_four_train,
    constraint_validation=constraints_four_validation,
    constraint_loss_fn=four_normalization_loss,
    lambda_norm=LAMBDA_NORM_4,
)


## Four-class posterior closure

The structured heads provide a conditionally normalized hierarchical posterior,

$$
\widehat p_H(\mu,\alpha\mid x)=
\frac{q_P(\mu\mid x)e^{a(\mu,x)}}{Z_P(x)}
\frac{q_N(\alpha\mid x,\mu)e^{b(\mu,\alpha,x)}}{Z_N(x,\mu)}.
$$

This uses only proposal Monte Carlo normalizers.  We compare it with both the compositionally equivalent population expression $q_Pq_Ne^{a+b}/Z_{PN}$ and the independently normalized likelihood route

$$
\widehat p_L(\mu,\alpha\mid x)\propto
\rho_\mu\rho_\alpha q_L(x\mid\mu,\alpha)e^c/Z_L(\mu,\alpha).
$$

At finite capacity the hierarchical and composed routes can differ; that difference is a diagnostic, not an invitation to multiply in another ad hoc normalization factor.  Analytic contours remain validation-only.


In [ ]:
def four_log_zp(ensemble, x_values, n_reference, seed):
    x_values = np.atleast_2d(np.asarray(x_values, dtype=np.float32))
    mu = _draw_conditional(q_p, x_values, n_reference, seed)
    alpha_filler = np.zeros_like(mu)
    points = np.concatenate([
        mu, alpha_filler,
        np.repeat(x_values[:, None, :], n_reference, axis=1),
    ], axis=2)
    logits = predict_shared_logits(ensemble, points)
    return logsumexp(logits[..., 1] - logits[..., 2], axis=1) - np.log(n_reference)


def four_log_zl(ensemble, theta_values, n_reference, seed):
    theta_values = np.atleast_2d(np.asarray(theta_values, dtype=np.float32))
    # Stream conditioning contexts.  A paper grid has O(10^4) theta
    # values, so materializing every theta x reference draw at once would
    # create multi-GB ensemble-logit temporaries.
    context_batch = max(1, 65_536 // int(n_reference))
    chunks = []
    for start in range(0, len(theta_values), context_batch):
        theta_chunk = theta_values[start : start + context_batch]
        x = _draw_conditional(
            q_l, theta_chunk, n_reference, seed + start
        )
        points = np.concatenate([
            np.repeat(theta_chunk[:, None, :], n_reference, axis=1), x
        ], axis=2)
        logits = predict_shared_logits(ensemble, points)
        chunks.append(
            logsumexp(logits[..., 0] - logits[..., 3], axis=1)
            - np.log(n_reference)
        )
    return np.concatenate(chunks)

def four_log_zn(ensemble, x_values, mu_values, n_reference, seed):
    x_values = np.atleast_2d(np.asarray(x_values, dtype=np.float32))
    mu_values = np.asarray(mu_values, dtype=np.float32).reshape(-1, 1)
    if len(x_values) == 1 and len(mu_values) > 1:
        x_values = np.repeat(x_values, len(mu_values), axis=0)
    alpha = _draw_conditional(
        q_n, _qn_context(x_values, mu_values), n_reference, seed
    )
    points = np.concatenate([
        np.repeat(mu_values[:, None, :], n_reference, axis=1),
        alpha,
        np.repeat(x_values[:, None, :], n_reference, axis=1),
    ], axis=2)
    logits = predict_shared_logits(ensemble, points)
    return logsumexp(logits[..., 0] - logits[..., 1], axis=1) - np.log(n_reference)

def four_log_zpn(ensemble, x_values, n_reference, seed):
    x_values = np.atleast_2d(np.asarray(x_values, dtype=np.float32))
    mu, alpha = _draw_joint_reference(x_values, n_reference, seed)
    points = np.concatenate([
        mu, alpha, np.repeat(x_values[:, None, :], n_reference, axis=1)
    ], axis=2)
    logits = predict_shared_logits(ensemble, points)
    return logsumexp(logits[..., 0] - logits[..., 2], axis=1) - np.log(n_reference)

MU_GRID_2D = np.linspace(-3.5, 3.5, 181 if not SMOKE_MODE else 71)
ALPHA_GRID_2D = np.linspace(-3.2, 3.2, 161 if not SMOKE_MODE else 61)
MU_MESH, ALPHA_MESH = np.meshgrid(MU_GRID_2D, ALPHA_GRID_2D, indexing="ij")
THETA_GRID = np.column_stack([MU_MESH.ravel(), ALPHA_MESH.ravel()])
X_GRID = np.repeat(X_OBS[None, :], len(THETA_GRID), axis=0)
POINTS_GRID = np.column_stack([THETA_GRID, X_GRID]).astype(np.float32)

logits_grid = predict_shared_logits(four_normalized, POINTS_GRID)

log_qp_grid = spline_flow_log_prob(
    q_p, THETA_GRID[:, :1], context=X_GRID
)
log_qn_grid = spline_flow_log_prob(
    q_n,
    THETA_GRID[:, 1:2],
    context=_qn_context(X_GRID, THETA_GRID[:, :1]),
)

log_zp_observed = float(four_log_zp(
    four_normalized, X_OBS, N_GRID_REFERENCE, SEED + 990
)[0])
log_zn_mu = four_log_zn(
    four_normalized,
    X_OBS,
    MU_GRID_2D,
    N_GRID_REFERENCE,
    SEED + 991,
)
log_zn_grid = np.repeat(log_zn_mu, len(ALPHA_GRID_2D))
log_joint_hierarchical = (
    log_qp_grid + logits_grid[:, 1] - logits_grid[:, 2] - log_zp_observed
    + log_qn_grid + logits_grid[:, 0] - logits_grid[:, 1] - log_zn_grid
)
posterior_four_hierarchical, _ = normalize_log_surface(
    log_joint_hierarchical.reshape(MU_MESH.shape), MU_GRID_2D, ALPHA_GRID_2D
)

log_zpn_observed = float(four_log_zpn(
    four_normalized, X_OBS, N_GRID_REFERENCE, SEED + 992
)[0])
log_joint_composed = (
    log_qp_grid + log_qn_grid
    + logits_grid[:, 0] - logits_grid[:, 2] - log_zpn_observed
)
posterior_four_composed, _ = normalize_log_surface(
    log_joint_composed.reshape(MU_MESH.shape), MU_GRID_2D, ALPHA_GRID_2D
)


log_zl_grid = four_log_zl(
    four_normalized, THETA_GRID, N_GRID_REFERENCE, SEED + 1000
)
log_ql_grid = spline_flow_log_prob(q_l, X_GRID, context=THETA_GRID)
log_joint_likelihood = (
    design_logpdf(THETA_GRID)
    + log_ql_grid
    + logits_grid[:, 0]
    - logits_grid[:, 3]
    - log_zl_grid
)
posterior_four_likelihood, _ = normalize_log_surface(
    log_joint_likelihood.reshape(MU_MESH.shape), MU_GRID_2D, ALPHA_GRID_2D
)

log_joint_truth = design_logpdf(THETA_GRID) + log_likelihood(X_OBS, THETA_GRID)
posterior_truth_2d, log_evidence_grid_truth = normalize_log_surface(
    log_joint_truth.reshape(MU_MESH.shape), MU_GRID_2D, ALPHA_GRID_2D
)
truth_mu_2d = np.trapezoid(posterior_truth_2d, ALPHA_GRID_2D, axis=1)
truth_alpha_2d = np.trapezoid(posterior_truth_2d, MU_GRID_2D, axis=0)
hierarchical_mu_2d = np.trapezoid(
    posterior_four_hierarchical, ALPHA_GRID_2D, axis=1
)
hierarchical_alpha_2d = np.trapezoid(
    posterior_four_hierarchical, MU_GRID_2D, axis=0
)
composed_mu_2d = np.trapezoid(posterior_four_composed, ALPHA_GRID_2D, axis=1)
composed_alpha_2d = np.trapezoid(posterior_four_composed, MU_GRID_2D, axis=0)
likelihood_mu_2d = np.trapezoid(posterior_four_likelihood, ALPHA_GRID_2D, axis=1)
likelihood_alpha_2d = np.trapezoid(posterior_four_likelihood, MU_GRID_2D, axis=0)
print(
    "Posterior-grid evidence minus wide 1D truth =",
    f"{log_evidence_grid_truth - LOG_EVIDENCE_TRUTH:+.4f}",
    "(finite plotting-window truncation)",
)
four_summary = pd.DataFrame([
    {
        "route": "hierarchically normalized posterior",
        "2D JS distance": js_distance_discrete(
            posterior_truth_2d, posterior_four_hierarchical
        ),
        "mu marginal IAE": integrated_absolute_error(
            truth_mu_2d, hierarchical_mu_2d, MU_GRID_2D
        ),
        "alpha marginal IAE": integrated_absolute_error(
            truth_alpha_2d, hierarchical_alpha_2d, ALPHA_GRID_2D
        ),
    },
    {
        "route": "composed S/P diagnostic",
        "2D JS distance": js_distance_discrete(
            posterior_truth_2d, posterior_four_composed
        ),
        "mu marginal IAE": integrated_absolute_error(
            truth_mu_2d, composed_mu_2d, MU_GRID_2D
        ),
        "alpha marginal IAE": integrated_absolute_error(
            truth_alpha_2d, composed_alpha_2d, ALPHA_GRID_2D
        ),
    },
    {
        "route": "normalized likelihood",
        "2D JS distance": js_distance_discrete(
            posterior_truth_2d, posterior_four_likelihood
        ),
        "mu marginal IAE": integrated_absolute_error(
            truth_mu_2d, likelihood_mu_2d, MU_GRID_2D
        ),
        "alpha marginal IAE": integrated_absolute_error(
            truth_alpha_2d, likelihood_alpha_2d, ALPHA_GRID_2D
        ),
    },
])
display(four_summary.style.format(precision=4))


In [ ]:
def highest_density_levels(density, x_grid, y_grid, masses=(0.5, 0.9)):
    density = np.asarray(density, dtype=float)
    dx = float(np.mean(np.diff(x_grid)))
    dy = float(np.mean(np.diff(y_grid)))
    ordered = np.sort(density.ravel())[::-1]
    cumulative = np.cumsum(ordered) * dx * dy
    return sorted([
        ordered[min(np.searchsorted(cumulative, mass), len(ordered) - 1)]
        for mass in masses
    ])

truth_levels = highest_density_levels(
    posterior_truth_2d, MU_GRID_2D, ALPHA_GRID_2D
)
hierarchical_levels = highest_density_levels(
    posterior_four_hierarchical, MU_GRID_2D, ALPHA_GRID_2D
)
likelihood_levels = highest_density_levels(
    posterior_four_likelihood, MU_GRID_2D, ALPHA_GRID_2D
)

fig, axes = plt.subplots(2, 2, figsize=(11.8, 9.0), constrained_layout=True)
axes[0, 0].contour(
    MU_GRID_2D, ALPHA_GRID_2D, posterior_truth_2d.T,
    levels=truth_levels, colors="black", linewidths=[2.2, 1.5],
)
axes[0, 0].contour(
    MU_GRID_2D, ALPHA_GRID_2D, posterior_four_hierarchical.T,
    levels=hierarchical_levels, colors=FULL4_COLOR, linewidths=[2.2, 1.5], linestyles="--",
)
axes[0, 0].set_title("(a) Joint posterior: hierarchical route\nblack truth; green learned")

axes[0, 1].contour(
    MU_GRID_2D, ALPHA_GRID_2D, posterior_truth_2d.T,
    levels=truth_levels, colors="black", linewidths=[2.2, 1.5],
)
axes[0, 1].contour(
    MU_GRID_2D, ALPHA_GRID_2D, posterior_four_likelihood.T,
    levels=likelihood_levels, colors=FULL4_COLOR, linewidths=[2.2, 1.5], linestyles="--",
)
axes[0, 1].set_title("(b) Joint posterior: likelihood route\nblack truth; green learned")
for ax in axes[0]:
    ax.set(xlabel=r"$\mu$", ylabel=r"$\alpha$")
    ax.grid(alpha=0.2)

three_normalized_interp = np.interp(MU_GRID_2D, MU_GRID, closure_normalized["posterior"])
axes[1, 0].plot(MU_GRID_2D, truth_mu_2d, color="black", lw=2.3, label="analytic truth")
axes[1, 0].plot(
    MU_GRID_2D, three_normalized_interp, color=FULL3_COLOR, lw=1.8,
    label="three-class (alpha hidden)",
)
axes[1, 0].plot(
    MU_GRID_2D, hierarchical_mu_2d, color=FULL4_COLOR, lw=2.1,
    label="four-class hierarchical",
)
axes[1, 0].plot(
    MU_GRID_2D, composed_mu_2d, color=FULL4_COLOR, lw=1.5, ls=":",
    label="composed S/P diagnostic",
)
axes[1, 0].plot(
    MU_GRID_2D, likelihood_mu_2d, color=FULL4_COLOR, lw=1.8, ls="--",
    label="four-class likelihood",
)
axes[1, 0].set(xlabel=r"$\mu$", ylabel="marginal posterior", title="(c) Parameter of interest")
axes[1, 0].legend(fontsize=8.5)

axes[1, 1].plot(ALPHA_GRID_2D, truth_alpha_2d, color="black", lw=2.3, label="analytic truth")
axes[1, 1].plot(
    ALPHA_GRID_2D, hierarchical_alpha_2d, color=FULL4_COLOR, lw=2.1,
    label="four-class hierarchical",
)
axes[1, 1].plot(
    ALPHA_GRID_2D, composed_alpha_2d, color=FULL4_COLOR, lw=1.5, ls=":",
    label="composed S/P diagnostic",
)
axes[1, 1].plot(
    ALPHA_GRID_2D, likelihood_alpha_2d, color=FULL4_COLOR, lw=1.8, ls="--",
    label="four-class likelihood",
)
axes[1, 1].set(xlabel=r"$\alpha$", ylabel="marginal posterior", title="(d) Systematic nuisance")
axes[1, 1].legend(fontsize=8.5)
for ax in axes[1]:
    ax.grid(alpha=0.25)

export_exercise9_multiclass_figure(fig, "four_class_joint_posterior_closure")
plt.show()


## Held-out calibration with the nuisance explicit

A single observed-data contour is not an amortized calibration test.  On fresh simulator draws we therefore estimate marginal PIT and equal-tailed coverage for both $\mu$ and $\alpha$.

For every held-out $x$, the calculation draws $\mu\sim q_P$, reweights it by $e^a$, then draws $\alpha\sim q_N$ at each sampled $\mu$ and conditionally reweights by $e^b$.  Normalizing those sample weights within each conditional level implements the hierarchical posterior without evaluating the analytic likelihood, marginal likelihood, or any truth-density residual.  The raw $q_Pq_N$ hierarchy is shown as a proposal baseline.


In [ ]:
def four_class_calibration_pits(
    ensemble,
    theta_true,
    x_true,
    n_mu,
    n_alpha,
    seed,
    context_batch=8,
):
    theta_true = np.asarray(theta_true, dtype=np.float32)
    x_true = np.asarray(x_true, dtype=np.float32)
    learned_mu, learned_alpha = [], []
    proposal_mu, proposal_alpha = [], []
    for start in range(0, len(theta_true), int(context_batch)):
        stop = min(len(theta_true), start + int(context_batch))
        theta_batch = theta_true[start:stop]
        x_batch = x_true[start:stop]
        batch_size = len(x_batch)

        mu = _draw_conditional(q_p, x_batch, n_mu, seed + 10 * start)
        x_at_mu = np.repeat(x_batch[:, None, :], n_mu, axis=1)
        a_points = np.concatenate([mu, np.zeros_like(mu), x_at_mu], axis=2)
        a_logits = predict_shared_logits(ensemble, a_points)
        log_a = a_logits[..., 1] - a_logits[..., 2]
        mu_weights = np.exp(log_a - logsumexp(log_a, axis=1, keepdims=True))
        mu_indicator = mu[..., 0] <= theta_batch[:, 0, None]
        learned_mu.append(np.sum(mu_weights * mu_indicator, axis=1))
        proposal_mu.append(mu_indicator.mean(axis=1))

        flat_mu = mu.reshape(-1, 1)
        flat_x = x_at_mu.reshape(-1, 3)
        alpha = _draw_conditional(
            q_n,
            _qn_context(flat_x, flat_mu),
            n_alpha,
            seed + 10 * start + 1,
        )
        mu_at_alpha = np.repeat(flat_mu[:, None, :], n_alpha, axis=1)
        x_at_alpha = np.repeat(flat_x[:, None, :], n_alpha, axis=1)
        b_points = np.concatenate([mu_at_alpha, alpha, x_at_alpha], axis=2)
        b_logits = predict_shared_logits(ensemble, b_points)
        log_b = b_logits[..., 0] - b_logits[..., 1]
        alpha_weights = np.exp(
            log_b - logsumexp(log_b, axis=1, keepdims=True)
        )
        alpha_truth = np.repeat(theta_batch[:, 1], n_mu)
        alpha_indicator = alpha[..., 0] <= alpha_truth[:, None]
        conditional_cdf = np.sum(alpha_weights * alpha_indicator, axis=1)
        learned_alpha.append(
            np.sum(mu_weights * conditional_cdf.reshape(batch_size, n_mu), axis=1)
        )
        proposal_alpha.append(
            alpha_indicator.reshape(batch_size, n_mu, n_alpha).mean(axis=(1, 2))
        )

    return {
        "proposal q_P q_N": {
            "mu": np.concatenate(proposal_mu),
            "alpha": np.concatenate(proposal_alpha),
        },
        "structured hierarchical": {
            "mu": np.concatenate(learned_mu),
            "alpha": np.concatenate(learned_alpha),
        },
    }


rng = np.random.default_rng(SEED + 1400)
theta_four_calibration = sample_design(N_FOUR_CALIBRATION_CONTEXTS, rng)
x_four_calibration = simulate(theta_four_calibration, rng)
four_pit_values = four_class_calibration_pits(
    four_normalized,
    theta_four_calibration,
    x_four_calibration,
    N_FOUR_CALIBRATION_MU,
    N_FOUR_CALIBRATION_ALPHA,
    SEED + 1401,
)

FOUR_NOMINAL_COVERAGE = np.linspace(0.05, 0.95, 19)
four_coverage = {
    method: {
        parameter: np.array([
            np.mean(
                (pit >= 0.5 * (1.0 - level))
                & (pit <= 0.5 * (1.0 + level))
            )
            for level in FOUR_NOMINAL_COVERAGE
        ])
        for parameter, pit in values.items()
    }
    for method, values in four_pit_values.items()
}
four_calibration_rows = []
for method, values in four_pit_values.items():
    for parameter, pit in values.items():
        ordered = np.sort(pit)
        uniform = (np.arange(len(ordered)) + 0.5) / len(ordered)
        four_calibration_rows.append({
            "method": method,
            "parameter": parameter,
            "PIT KS distance": float(np.max(np.abs(ordered - uniform))),
            "max coverage error": float(np.max(np.abs(
                four_coverage[method][parameter] - FOUR_NOMINAL_COVERAGE
            ))),
        })
four_calibration_summary = pd.DataFrame(four_calibration_rows)
display(four_calibration_summary.style.format(precision=4))

fig, axes = plt.subplots(2, 2, figsize=(11.0, 8.4), constrained_layout=True)
method_colors = {
    "proposal q_P q_N": "0.55",
    "structured hierarchical": FULL4_COLOR,
}
parameter_symbols = {"mu": r"\mu", "alpha": r"\alpha"}
for column, parameter in enumerate(["mu", "alpha"]):
    symbol = parameter_symbols[parameter]
    for method, values in four_pit_values.items():
        pit = values[parameter]
        ordered = np.sort(pit)
        empirical_cdf = np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
        axes[0, column].plot(
            ordered, empirical_cdf, lw=1.9,
            color=method_colors[method], label=method,
        )
        axes[1, column].plot(
            FOUR_NOMINAL_COVERAGE,
            four_coverage[method][parameter],
            lw=1.9,
            color=method_colors[method],
            label=method,
        )
    axes[0, column].plot([0, 1], [0, 1], color="black", ls="--", lw=1)
    axes[1, column].plot([0, 1], [0, 1], color="black", ls="--", lw=1)
    coverage_sigma = np.sqrt(
        FOUR_NOMINAL_COVERAGE * (1.0 - FOUR_NOMINAL_COVERAGE)
        / N_FOUR_CALIBRATION_CONTEXTS
    )
    axes[1, column].fill_between(
        FOUR_NOMINAL_COVERAGE,
        np.maximum(0.0, FOUR_NOMINAL_COVERAGE - coverage_sigma),
        np.minimum(1.0, FOUR_NOMINAL_COVERAGE + coverage_sigma),
        color="0.7", alpha=0.2, linewidth=0, label=r"$\pm1\sigma$ binomial",
    )
    axes[0, column].set(
        xlabel=rf"${symbol}$ posterior PIT",
        ylabel="empirical CDF",
        title=rf"({chr(97 + column)}) ${symbol}$ PIT",
    )
    axes[1, column].set(
        xlabel="nominal equal-tailed coverage",
        ylabel="empirical coverage",
        title=rf"({chr(99 + column)}) ${symbol}$ coverage",
    )
    for row in range(2):
        axes[row, column].set(xlim=(0, 1), ylim=(0, 1))
        axes[row, column].grid(alpha=0.25)
        axes[row, column].legend(fontsize=8.5)

export_exercise9_multiclass_figure(fig, "four_class_heldout_calibration")
plt.show()


## Held-out nuisance and evidence consistency—not a loss

The next figure reconstructs $p_m(x_{\rm obs}\mid\mu)$ while deliberately varying $\alpha$, and reconstructs $m_\rho(x_{\rm obs})$ along paths through $(\mu,\alpha)$.  Fresh proposal samples estimate every displayed partition.

These curves never enter the objective.  They test whether shared multiclass CE, structured ratio composition, and conditional mass normalization are sufficient to produce Bayes-compatible routes.  Analytic densities are used only to place the zero-reference line.  Departures in regions where the fixed observation and parameter have negligible joint support are extrapolation diagnostics and are interpreted separately from posterior-supported closure.


In [ ]:
def nuisance_consistency_curve(ensemble, mu_value, alpha_grid, x_observed, n_reference, seed):
    alpha_grid = np.asarray(alpha_grid, dtype=float)
    theta = np.column_stack([
        np.full_like(alpha_grid, float(mu_value)), alpha_grid
    ]).astype(np.float32)
    x = np.repeat(np.asarray(x_observed)[None, :], len(theta), axis=0).astype(np.float32)
    points = np.column_stack([theta, x])
    logits = predict_shared_logits(ensemble, points)
    log_ql = spline_flow_log_prob(q_l, x, context=theta)
    log_qn = spline_flow_log_prob(
        q_n, theta[:, 1:2], context=_qn_context(x, theta[:, :1])
    )
    log_zn = float(four_log_zn(
        ensemble, x_observed, [mu_value], n_reference, seed
    )[0])
    log_zl = four_log_zl(ensemble, theta, n_reference, seed + 1)
    raw = design_alpha_logpdf(alpha_grid) + log_ql - log_qn + logits[:, 1] - logits[:, 3]
    normalized = raw + log_zn - log_zl
    truth = float(marginal_log_likelihood(x_observed, [mu_value])[0])
    return raw - truth, normalized - truth

def evidence_consistency_path(ensemble, theta, x_observed, n_reference, seed):
    theta = np.atleast_2d(np.asarray(theta, dtype=np.float32))
    x = np.repeat(np.asarray(x_observed)[None, :], len(theta), axis=0).astype(np.float32)
    points = np.column_stack([theta, x])
    logits = predict_shared_logits(ensemble, points)
    log_qp = spline_flow_log_prob(q_p, theta[:, :1], context=x)
    log_qn = spline_flow_log_prob(
        q_n, theta[:, 1:2], context=_qn_context(x, theta[:, :1])
    )
    log_ql = spline_flow_log_prob(q_l, x, context=theta)
    log_zpn = float(four_log_zpn(
        ensemble, x_observed, n_reference, seed
    )[0])
    log_zl = four_log_zl(ensemble, theta, n_reference, seed + 1)
    raw = (
        design_logpdf(theta) + log_ql - log_qp - log_qn
        + logits[:, 2] - logits[:, 3]
    )
    normalized = raw + log_zpn - log_zl
    return raw - LOG_EVIDENCE_TRUTH, normalized - LOG_EVIDENCE_TRUTH

# Locate the two most separated posterior modes for illustrative slices.
local_peak = np.r_[False, (truth_mu[1:-1] > truth_mu[:-2]) & (truth_mu[1:-1] > truth_mu[2:]), False]
peak_candidates = MU_GRID[local_peak]
peak_heights = truth_mu[local_peak]
if len(peak_candidates) >= 2:
    selected = np.argsort(peak_heights)[-2:]
    MU_SLICES = np.sort(peak_candidates[selected])
else:
    MU_SLICES = np.array([-1.4, 1.2])

ALPHA_CONSISTENCY_GRID = np.linspace(-2.6, 2.6, 81 if not SMOKE_MODE else 31)
nuisance_curves = {}
for index, mu_value in enumerate(MU_SLICES):
    nuisance_curves[float(mu_value)] = nuisance_consistency_curve(
        four_normalized,
        mu_value,
        ALPHA_CONSISTENCY_GRID,
        X_OBS,
        N_DIAGNOSTIC_REFERENCE,
        SEED + 1100 + 10 * index,
    )

MU_PATH = np.linspace(-3.0, 3.0, 101 if not SMOKE_MODE else 41)
THETA_MU_PATH = np.column_stack([MU_PATH, np.zeros_like(MU_PATH)])
raw_mu_path, normalized_mu_path = evidence_consistency_path(
    four_normalized, THETA_MU_PATH, X_OBS, N_DIAGNOSTIC_REFERENCE, SEED + 1150
)
ALPHA_PATH = np.linspace(-2.6, 2.6, 101 if not SMOKE_MODE else 41)
mu_mode = float(MU_GRID[np.argmax(truth_mu)])
THETA_ALPHA_PATH = np.column_stack([
    np.full_like(ALPHA_PATH, mu_mode), ALPHA_PATH
])
raw_alpha_path, normalized_alpha_path = evidence_consistency_path(
    four_normalized, THETA_ALPHA_PATH, X_OBS, N_DIAGNOSTIC_REFERENCE, SEED + 1160
)

# Held-out Z distributions from ordinary, fresh design anchors.
rng = np.random.default_rng(SEED + 1190)
n_z_contexts = max(48, n_context_check)
theta_z_check = sample_design(n_z_contexts, rng)
x_z_check = simulate(theta_z_check, rng)
n_z_reference = min(256, N_DIAGNOSTIC_REFERENCE)
heldout_four_z = {
    r"$Z_P$": four_log_zp(
        four_normalized, x_z_check, n_z_reference, SEED + 1191
    ),
    r"$Z_N$": four_log_zn(
        four_normalized,
        x_z_check,
        theta_z_check[:, 0],
        n_z_reference,
        SEED + 1192,
    ),
    r"$Z_L$": four_log_zl(
        four_normalized, theta_z_check, n_z_reference, SEED + 1193
    ),
    r"$Z_{PN}$": four_log_zpn(
        four_normalized, x_z_check, n_z_reference, SEED + 1194
    ),
}
for name, values in heldout_four_z.items():
    print(
        f"{name:8s}: median|log Z|={np.median(np.abs(values)):.3f}, "
        f"q95={np.quantile(np.abs(values), .95):.3f}, "
        f"max={np.max(np.abs(values)):.3f}, "
        f"RMS={np.sqrt(np.mean(values**2)):.3f}"
    )


n_tail_four = 2_000 if SMOKE_MODE else (20_000 if FAST_MODE else 80_000)
mu_joint_tail, alpha_joint_tail = _draw_joint_reference(
    X_OBS[None, :], n_tail_four, SEED + 1191
)
joint_tail_points = np.concatenate([
    mu_joint_tail,
    alpha_joint_tail,
    np.repeat(X_OBS[None, None, :], n_tail_four, axis=1),
], axis=2)[0]
joint_tail_logits = predict_shared_logits(four_normalized, joint_tail_points)

alpha_n_tail = _draw_conditional(
    q_n,
    _qn_context(X_OBS, [mu_mode]),
    n_tail_four,
    SEED + 1192,
)[0]
nuisance_tail_points = np.column_stack([
    np.full(n_tail_four, mu_mode),
    alpha_n_tail[:, 0],
    np.repeat(X_OBS[None, :], n_tail_four, axis=0),
])
nuisance_tail_logits = predict_shared_logits(four_normalized, nuisance_tail_points)

theta_mode = THETA_GRID[int(np.argmax(posterior_truth_2d))]
x_l_tail = _draw_conditional(
    q_l, theta_mode[None, :], n_tail_four, SEED + 1193
)[0]
likelihood_tail_points = np.column_stack([
    np.repeat(theta_mode[None, :], n_tail_four, axis=0), x_l_tail
])
likelihood_tail_logits = predict_shared_logits(four_normalized, likelihood_tail_points)

four_tail_log_ratios = {
    r"$r_P=N/P$": joint_tail_logits[:, 1] - joint_tail_logits[:, 2],
    r"$r_N=S/N$": nuisance_tail_logits[:, 0] - nuisance_tail_logits[:, 1],
    r"$r_L=S/L$": likelihood_tail_logits[:, 0] - likelihood_tail_logits[:, 3],
    r"$r_{PN}=S/P$": joint_tail_logits[:, 0] - joint_tail_logits[:, 2],
}
four_tail_rows = []
for ratio_name, log_ratio in four_tail_log_ratios.items():
    summary = importance_tail_summary(log_ratio)
    four_tail_rows.append({
        "partition ratio": ratio_name,
        "log mean ratio": float(logsumexp(log_ratio) - np.log(len(log_ratio))),
        "ESS fraction": summary["ESS_fraction"],
        "Pareto k": summary["pareto_k"],
        "max weight": summary["max_weight_fraction"],
    })
four_tail_summary = pd.DataFrame(four_tail_rows)
display(four_tail_summary.style.format(precision=4))

four_head_saturation = pd.DataFrame([
    {
        "head / proposal": "a = log(N/P), joint q_P q_N",
        "saturation fraction": head_saturation_fraction(
            four_normalized, joint_tail_points, "a"
        ),
    },
    {
        "head / proposal": "b = log(S/N), q_N",
        "saturation fraction": head_saturation_fraction(
            four_normalized, nuisance_tail_points, "b"
        ),
    },
    {
        "head / proposal": "c = log(S/L), q_L",
        "saturation fraction": head_saturation_fraction(
            four_normalized, likelihood_tail_points, "c"
        ),
    },
])
display(four_head_saturation.style.format(precision=6))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12.0, 8.7), constrained_layout=True)
colors = ["#CC79A7", FULL4_COLOR]
for color, (mu_value, (raw, normalized)) in zip(colors, nuisance_curves.items()):
    axes[0, 0].plot(
        ALPHA_CONSISTENCY_GRID, raw, color=color, lw=1.2, ls=":",
        label=rf"raw, $\mu={mu_value:+.2f}$",
    )
    axes[0, 0].plot(
        ALPHA_CONSISTENCY_GRID, normalized, color=color, lw=2.0,
        label=rf"normalized, $\mu={mu_value:+.2f}$",
    )
axes[0, 0].axhline(0.0, color="black", lw=1)
axes[0, 0].set(
    xlabel=r"$\alpha$ used in the reconstruction",
    ylabel=r"$\log\widehat p_m-\log p_m^{\rm true}$",
    title="(a) Nuisance-marginal consistency",
)
axes[0, 0].legend(fontsize=8)

axes[0, 1].plot(MU_PATH, raw_mu_path, color="0.55", lw=1.2, ls=":", label="raw")
axes[0, 1].plot(MU_PATH, normalized_mu_path, color=FULL4_COLOR, lw=2.0, label="normalized")
axes[0, 1].axhline(0.0, color="black", lw=1)
axes[0, 1].set(
    xlabel=r"$\mu$ at $\alpha=0$", ylabel=r"$\log\widehat m-\log m^{\rm true}$",
    title="(b) Evidence consistency along a POI path",
)
axes[0, 1].legend(fontsize=8.5)

axes[1, 0].plot(ALPHA_PATH, raw_alpha_path, color="0.55", lw=1.2, ls=":", label="raw")
axes[1, 0].plot(ALPHA_PATH, normalized_alpha_path, color=FULL4_COLOR, lw=2.0, label="normalized")
axes[1, 0].axhline(0.0, color="black", lw=1)
axes[1, 0].set(
    xlabel=rf"$\alpha$ at $\mu={mu_mode:+.2f}$",
    ylabel=r"$\log\widehat m-\log m^{\rm true}$",
    title="(c) Evidence consistency along a nuisance path",
)
axes[1, 0].legend(fontsize=8.5)

for label, values in heldout_four_z.items():
    ordered = np.sort(values)
    cdf = np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
    axes[1, 1].plot(ordered, cdf, lw=1.8, label=label)
axes[1, 1].axvline(0.0, color="black", lw=1)
axes[1, 1].set(
    xlabel=r"held-out $\log Z$", ylabel="empirical CDF",
    title="(d) Four conditional partitions",
)
axes[1, 1].legend(fontsize=8.5)
for ax in axes.flat:
    ax.grid(alpha=0.25)

export_exercise9_multiclass_figure(fig, "four_class_consistency_and_normalization")
plt.show()


## A systematic-prior and auxiliary-measurement update

Because the four-class model retains $\alpha$, the same trained posterior can be updated without retraining.  We replace the nuisance design density by
$\pi_{\rm new}(\alpha)=\mathcal N(0.30,0.45^2)$ and include an auxiliary measurement
$a_{\rm obs}=0.10$ with $a\mid\alpha\sim\mathcal N(\alpha,0.25^2)$.

On the grid the learned joint density is multiplied by

$$
  \frac{\pi_{\rm new}(\alpha)}{\rho_\alpha(\alpha)}p(a_{\rm obs}\mid\alpha).
$$

This cell is an application check, not part of classifier training.


In [ ]:
ALPHA_PRIOR_MEAN, ALPHA_PRIOR_SIGMA = 0.30, 0.45
A_OBSERVED, SIGMA_A = 0.10, 0.25
log_update_factor = (
    norm.logpdf(
        THETA_GRID[:, 1], loc=ALPHA_PRIOR_MEAN, scale=ALPHA_PRIOR_SIGMA
    )
    - design_alpha_logpdf(THETA_GRID[:, 1])
    + norm.logpdf(A_OBSERVED, loc=THETA_GRID[:, 1], scale=SIGMA_A)
).reshape(MU_MESH.shape)
learned_updated, _ = normalize_log_surface(
    np.log(np.maximum(posterior_four_hierarchical, 1.0e-300)) + log_update_factor,
    MU_GRID_2D,
    ALPHA_GRID_2D,
)
truth_updated, _ = normalize_log_surface(
    log_joint_truth.reshape(MU_MESH.shape) + log_update_factor,
    MU_GRID_2D,
    ALPHA_GRID_2D,
)
learned_update_mu = np.trapezoid(learned_updated, ALPHA_GRID_2D, axis=1)
learned_update_alpha = np.trapezoid(learned_updated, MU_GRID_2D, axis=0)
truth_update_mu = np.trapezoid(truth_updated, ALPHA_GRID_2D, axis=1)
truth_update_alpha = np.trapezoid(truth_updated, MU_GRID_2D, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(11.3, 4.3), constrained_layout=True)
axes[0].plot(MU_GRID_2D, truth_update_mu, color="black", lw=2.3, label="updated truth")
axes[0].plot(MU_GRID_2D, learned_update_mu, color=FULL4_COLOR, lw=2.0, label="updated hierarchical four-class")
axes[0].plot(MU_GRID_2D, truth_mu_2d, color="0.6", ls="--", label="baseline truth")
axes[0].set(xlabel=r"$\mu$", ylabel="posterior density", title="(a) Updated POI")
axes[1].plot(ALPHA_GRID_2D, truth_update_alpha, color="black", lw=2.3, label="updated truth")
axes[1].plot(ALPHA_GRID_2D, learned_update_alpha, color=FULL4_COLOR, lw=2.0, label="updated hierarchical four-class")
axes[1].plot(ALPHA_GRID_2D, truth_alpha_2d, color="0.6", ls="--", label="baseline truth")
axes[1].set(xlabel=r"$\alpha$", ylabel="posterior density", title="(b) Updated systematic")
for ax in axes:
    ax.legend(fontsize=8.5)
    ax.grid(alpha=0.25)
export_exercise9_multiclass_figure(fig, "four_class_systematic_update")
plt.show()

print(
    "Total inverse-RQS kernel calls retried in float64 in this run:",
    _rqs_retry_count(),
)


## Conclusions and paper-run checklist

This refactored exercise tests a narrower and cleaner claim:

1. A single multiclass discriminator can learn simulator-anchored posterior, conditional-nuisance, and likelihood residuals from samples.
2. The three-class CE-only versus CE-plus-normalization comparison isolates the value of conditional mass constraints.  There is no separate bridge-consistency penalty.
3. Independent A/B proposal halves give an unbiased cross-moment estimator of $(Z-1)^2$ for a fixed model; the same cross moment selects checkpoints, while positive pooled mass error is reported only as a variance-sensitive diagnostic. Because a finite set of banks is reused adaptively, final claims rely on wholly fresh post-training closure draws.
4. The four-class ratio-head structure enforces the exact $\alpha$ cancellation in $N/P$ and composes $S/P=(N/P)(S/N)$ without using analytic densities.
5. Nuisance-marginal and evidence consistency remain held-out tests.  Agreement of two learned routes is not a substitute for simulator closure, PIT/coverage, or proposal-tail diagnostics.

Before using the figures in the paper:

- run full mode on a GPU with `LOAD_IF_AVAILABLE=False` after any architecture or loss change;
- inspect the sample-only $q_L^m/q_L$ marginal, sliced-Wasserstein, correlation, and extreme-quantile audits before classifier results;
- report median, q95, and maximum $|\log Z|$ separately, plus ESS/Pareto-$k$ and head saturation;
- compare the hierarchical, composed, and likelihood posterior routes, and report the held-out $\mu/\alpha$ PIT and coverage figure;
- repeat complete flow/classifier training over several seeds and show paired bands;
- rerun with independently regenerated normalization banks and scan their inner sample size;
- treat improvement from normalization as an empirical result, never a consequence of the algebra;
- use the exported scripts and figures under `exercise9_multiclass_v2_figures_scripts/full/`.

For the paper, the strongest external baseline remains two independently trained binary posterior/likelihood residual classifiers.  The shared structured multiclass construction should be compared against that genuinely independent baseline, rather than against another shared model with an extra consistency penalty.
